# 예금보험공사 RAG 최종 크롤링 파이프라인

이 노트북은 42개 URL별 정책과 지금까지의 모든 피드백을 통합한 버전입니다.

## 미디어 정책

- **착오송금 제도 소개 영상(MT-001)**: 다운로드·전사·RAG 인덱싱하지 않고 공식 게시 페이지 참고 링크만 저장
- **착오송금 절차 웹툰(MT-009)**: 이미지 다운로드·OCR·RAG 인덱싱하지 않고 공식 게시 페이지 참고 링크만 저장
- 일반 이미지는 장식 요소를 제외하고 핵심 이미지 메타데이터만 저장하며 기본적으로 다운로드하지 않음

## 첨부파일 정책

- 공개 첨부파일은 백엔드 검수용 원본으로 다운로드
- MIME·파일 시그니처·SHA-256을 검증
- 안내서·절차·목록처럼 설명 가치가 있는 파일만 RAG 인덱싱
- 신청서·동의서·위임장·빈 양식은 다운로드만 하고 RAG에는 넣지 않음
- 사용자에게는 내부 저장 파일이 아니라 안정적인 공식 파일 URL을 제공
- 고정 URL이 없는 `gfn_downloadFile()` 방식은 공식 게시 페이지를 제공

로그인·본인인증·개인정보 조회·신청서 제출은 자동 실행하지 않습니다.

## 1. 라이브러리 설치

In [ ]:
%pip -q install "requests>=2.32,<3" "beautifulsoup4>=4.12,<5" "lxml>=5,<7" "pandas>=2.2,<4" "tqdm>=4.66,<5" "pypdf>=5,<7" "olefile>=0.47,<1"

## 2. 최종 크롤링 모듈 생성

In [ ]:
%%writefile kdic_final_pipeline.py
from __future__ import annotations

import hashlib
import json
import mimetypes
import re
import shutil
import subprocess
import sys
import time
from dataclasses import asdict, dataclass
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any, Iterable
from urllib.parse import parse_qs, urlencode, urljoin, urlparse, urlunparse
from urllib.robotparser import RobotFileParser

import pandas as pd
import requests
from bs4 import BeautifulSoup, Comment, NavigableString, Tag
from requests.adapters import HTTPAdapter
from tqdm.auto import tqdm
from urllib3.util.retry import Retry

KST = timezone(timedelta(hours=9))


# ============================================================
# 설정
# ============================================================

@dataclass
class PipelineConfig:
    select_business_functions: list[str] | None = None
    run_only_url_ids: list[str] | None = None
    max_urls: int | None = None

    request_delay_seconds: float = 1.2
    request_timeout_seconds: int = 30
    max_retries: int = 3
    max_asset_bytes: int = 50 * 1024 * 1024

    respect_robots_txt: bool = True
    enable_generic_pagination: bool = True
    pagination_max_pages: int = 100
    pagination_page_size: int = 10

    download_direct_attachments: bool = True
    download_images: bool = False
    download_js_attachments_with_playwright: bool = False

    # 영상·웹툰은 원본 파일을 내려받지 않고 공식 게시 페이지 링크만 제공합니다.
    collect_supplementary_media_links: bool = True
    download_videos: bool = False
    download_webtoons: bool = False

    # 첨부파일 하이브리드 정책
    extract_attachment_text: bool = True
    create_attachment_documents: bool = True
    attachment_index_min_chars: int = 120
    attachment_max_text_chars: int = 1_000_000
    attachment_pdf_max_pages: int = 300
    attachment_keep_failed_metadata: bool = True

    enable_playwright_snapshot: bool = False
    playwright_wait_ms: int = 1500

    create_chunks: bool = True
    chunk_max_chars: int = 1600
    min_chunk_chars: int = 80

    user_agent: str = (
        "KDIC-RAG-Academic-Crawler/1.0 "
        "(low-rate public-document collection; no authentication automation)"
    )

    def __post_init__(self) -> None:
        self.select_business_functions = self.select_business_functions or []
        self.run_only_url_ids = self.run_only_url_ids or []


@dataclass
class OutputPaths:
    root: Path
    raw_html: Path
    raw_assets: Path
    response_meta: Path
    parsed_markdown: Path
    parsed_json: Path
    parsed_attachments: Path
    pagination: Path
    dynamic_diagnostics: Path
    screenshots: Path
    logs: Path
    processed: Path
    manifest: Path
    quality: Path


def create_output_paths(runtime_root: Path) -> OutputPaths:
    run_id = datetime.now(KST).strftime("%Y%m%d_%H%M%S")
    root = runtime_root / f"kdic_crawl_output_{run_id}"
    paths = OutputPaths(
        root=root,
        raw_html=root / "raw" / "html",
        raw_assets=root / "raw" / "assets",
        response_meta=root / "raw" / "response_meta",
        parsed_markdown=root / "parsed" / "markdown",
        parsed_json=root / "parsed" / "json",
        parsed_attachments=root / "parsed" / "attachments",
        pagination=root / "diagnostics" / "pagination",
        dynamic_diagnostics=root / "diagnostics" / "dynamic",
        screenshots=root / "diagnostics" / "screenshots",
        logs=root / "logs",
        processed=root / "processed",
        manifest=root / "manifest",
        quality=root / "quality",
    )
    for value in asdict(paths).values():
        Path(value).mkdir(parents=True, exist_ok=True)
    return paths


# ============================================================
# 공통 유틸리티
# ============================================================

ATTACHMENT_EXTENSIONS = {
    ".pdf", ".hwp", ".hwpx", ".doc", ".docx", ".xls", ".xlsx",
    ".ppt", ".pptx", ".zip", ".csv", ".txt",
}
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".gif", ".webp", ".svg", ".bmp"}
VIDEO_EXTENSIONS = {".mp4", ".webm", ".mov", ".avi", ".mkv"}

NOISE_SELECTORS = [
    "script", "style", "noscript", "template", ".floatTop", ".floatBottom",
    ".paging", ".pagination", ".pageNavi", ".page_navi", ".sr-only",
    ".skip", ".skipnav", ".skip-navigation", ".loading", ".waitPage",
    "#waitPage", ".pdfViewerGuide", ".adobe-reader", ".tabMenu.clone",
]

NOISE_EXACT_TEXTS = {
    "퀵메뉴", "글자 크기", "글자확대", "글자축소", "KOR", "ENG",
    "상단으로 이동", "검색 초기화", "좌우로 움직여보세요", "표 더보기",
    "계산하기", "닫기", "Adobe Reader 바로가기",
    "영상내용입니다.", "영상 내용입니다.", "제도 소개 안내영상입니다.",
}

NOISE_CONTAINS_TEXTS = {
    "PDF파일 내용이 보이지 않으시면 Adobe Reader를 설치",
    "PDF 파일 내용이 보이지 않으시면 Adobe Reader를 설치",
    "영상내용입니다", "제도 소개 안내영상입니다",
}

ALLOWED_ACTION_DOMAIN_SUFFIXES = {
    "kdic.or.kr", "fins.kdic.or.kr", "ccrs.or.kr", "scourt.go.kr",
    "klac.or.kr", "law.go.kr", "easylaw.go.kr", "gov.kr",
}

ACTION_KEYWORDS = {
    "신청", "바로가기", "조회", "자가진단", "상담", "진행상황", "접수",
    "신고", "발급", "환급", "위치", "찾아오시는 길", "법령", "규정",
    "다운로드", "전화문의", "문의",
}

DOWNLOAD_CALL_RE = re.compile(
    r"gfn_downloadFile\(\s*['\"]([^'\"]+)['\"]\s*,\s*['\"]([^'\"]+)['\"]\s*\)"
)
URL_IN_JS_PATTERNS = [
    re.compile(r"gfn_openUrl\(\s*['\"]([^'\"]+)['\"]\s*\)"),
    re.compile(r"gfn_moveUrl\(\s*['\"]([^'\"]+)['\"]\s*\)"),
    re.compile(r"window\.open\(\s*['\"]([^'\"]+)['\"]"),
    re.compile(r"(?:window\.)?location(?:\.href)?\s*=\s*['\"]([^'\"]+)['\"]"),
    re.compile(r"location\.replace\(\s*['\"]([^'\"]+)['\"]\s*\)"),
]

BLOCK_TAGS = {
    "div", "section", "article", "aside", "header", "footer", "main",
    "p", "ul", "ol", "dl", "table", "figure", "h1", "h2", "h3",
    "h4", "h5", "h6",
}


def now_kst_iso() -> str:
    return datetime.now(KST).isoformat(timespec="seconds")


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def sha256_text(text: str) -> str:
    return sha256_bytes(text.encode("utf-8"))


def norm(text: str | None) -> str:
    return re.sub(r"\s+", " ", text or "").strip()


def safe_name(value: str, max_length: int = 100) -> str:
    value = re.sub(r'[\\/:*?"<>|]+', "_", norm(value))
    value = re.sub(r"_+", "_", value).strip("._ ")
    return (value or "untitled")[:max_length]


def ensure_parent(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)


def write_json(path: Path, data: Any) -> None:
    ensure_parent(path)
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")


def append_jsonl(path: Path, data: dict[str, Any]) -> None:
    ensure_parent(path)
    with path.open("a", encoding="utf-8") as file:
        file.write(json.dumps(data, ensure_ascii=False) + "\n")


def absolute_url(base_url: str, candidate: str | None, allow_contact: bool = False) -> str | None:
    if not candidate:
        return None
    candidate = candidate.strip()
    if not candidate or candidate.startswith(("#", "javascript:")):
        return None
    if candidate.startswith(("tel:", "mailto:")):
        return candidate if allow_contact else None
    return urljoin(base_url, candidate)


def is_noise_text(text: str) -> bool:
    value = norm(text)
    if not value:
        return True
    if value in NOISE_EXACT_TEXTS:
        return True
    return any(fragment in value for fragment in NOISE_CONTAINS_TEXTS)


def clean_visible_text(text: str) -> str:
    value = norm(text)
    for phrase in ["검색 초기화", "Adobe Reader 바로가기", "계산하기", "닫기"]:
        value = norm(value.replace(phrase, " "))
    for fragment in NOISE_CONTAINS_TEXTS:
        if fragment in value:
            value = norm(value.replace(fragment, " "))
    return value


def row_to_dict(row: pd.Series) -> dict[str, str]:
    return {str(k): "" if pd.isna(v) else str(v) for k, v in row.to_dict().items()}


def unique_dicts(items: Iterable[dict[str, Any]], keys: tuple[str, ...]) -> list[dict[str, Any]]:
    result: list[dict[str, Any]] = []
    seen: set[tuple[Any, ...]] = set()
    for item in items:
        key = tuple(item.get(name) for name in keys)
        if key in seen:
            continue
        seen.add(key)
        result.append(item)
    return result


# ============================================================
# 42개 검토 CSV → 실행 Manifest
# ============================================================

REVIEW_COLUMNS = {
    "문서_ID", "업무_도메인", "목표_도메인", "문서명", "페이지_유형",
    "권장_최종결정", "RAG_본문_인덱싱", "인덱싱_범위", "Action_링크",
    "권장_Action_Type", "Action_인증", "다중페이지_수집정책", "검토_의견",
    "검토_근거", "출처_URL",
    "첨부파일_원본수집", "첨부파일_RAG_정책", "첨부파일_사용자제공정책",
    "영상_처리정책", "웹툰_처리정책", "일반이미지_처리정책",
    "보조콘텐츠_표시조건", "보조콘텐츠_링크라벨",
}

RUN_DECISIONS = {
    "include_full", "include_reference", "include_full_public", "include_partial",
    "include_partial_action", "include_full_filtered", "include_separate_domain",
    "link_only",
}


def _site_name(url: str) -> str:
    host = (urlparse(url).hostname or "").lower()
    return "금융안심포털" if host.startswith("fins.") else "예금보험공사"


def _page_type(row: pd.Series) -> str:
    doc_id = row["문서_ID"]
    title = row["문서명"]
    policy = row["다중페이지_수집정책"]
    decision = row["권장_최종결정"]
    raw = row.get("페이지_유형", "")
    if decision == "link_only":
        return "link_only"
    if "FAQ" in title.upper() or "FAQ" in policy.upper():
        return "faq"
    if doc_id == "BI-004":
        return "dynamic_lookup"
    if "상호작용" in policy or "인증 서비스" in policy or "신청 서비스" in policy:
        return "interactive_service"
    return raw or "static_page"


def _fetch_strategy(row: pd.Series) -> str:
    decision = row["권장_최종결정"]
    policy = row["다중페이지_수집정책"]
    action = row["Action_링크"]
    if decision == "link_only":
        return "link_only"
    if "필수" in policy or "자동탐지" in policy:
        return "paginated_public"
    if "다운로드" in action:
        return "requests_html_assets"
    return "requests_html"


def _index_mode(value: str) -> str:
    if value == "O":
        return "full"
    if "공개 설명만" in value:
        return "public_only"
    if "보조 인덱스" in value:
        return "reference"
    if "일부 제외" in value:
        return "filtered"
    if "별도 도메인" in value:
        return "separate_domain"
    return "none"


def normalize_review_manifest(review_df: pd.DataFrame) -> pd.DataFrame:
    review_df = review_df.fillna("").astype(str)
    missing = sorted(REVIEW_COLUMNS - set(review_df.columns))
    if missing:
        raise ValueError(f"42개 검토 CSV 필수 열 누락: {missing}")

    records: list[dict[str, str]] = []
    for _, row in review_df.iterrows():
        url = row["출처_URL"]
        decision = row["권장_최종결정"]
        if decision not in RUN_DECISIONS:
            raise ValueError(f"지원하지 않는 결정: {row['문서_ID']}={decision}")
        page_type = _page_type(row)
        action_auth = row["Action_인증"]
        records.append({
            "url_id": row["문서_ID"],
            "business_function": row["업무_도메인"],
            "target_business_function": row["목표_도메인"] or row["업무_도메인"],
            "page_title": row["문서명"],
            "source_url": url,
            "site_name": _site_name(url),
            "normalized_decision": decision,
            "decision_reason": row["검토_의견"],
            "review_basis": row["검토_근거"],
            "page_type": page_type,
            "fetch_strategy": _fetch_strategy(row),
            "parser_profile": "kdic_final_media_attachment_v2",
            "auth_required": action_auth,
            "asset_policy": row["Action_링크"],
            "content_scope": row["인덱싱_범위"],
            "crawl_wave": (
                "META" if decision == "link_only" else
                "W2_DYNAMIC" if ("필수" in row["다중페이지_수집정책"] or row["문서_ID"] == "BI-004") else
                "W1_ASSET" if "다운로드" in row["Action_링크"] else
                "W1_STATIC"
            ),
            "rag_index_mode": _index_mode(row["RAG_본문_인덱싱"]),
            "rag_index_label": row["RAG_본문_인덱싱"],
            "action_policy": row["Action_링크"],
            "expected_action_types": row["권장_Action_Type"],
            "action_auth": action_auth,
            "pagination_policy": row["다중페이지_수집정책"],
            "attachment_download_policy": row["첨부파일_원본수집"],
            "attachment_rag_policy": row["첨부파일_RAG_정책"],
            "attachment_user_delivery_policy": row["첨부파일_사용자제공정책"],
            "video_policy": row["영상_처리정책"],
            "webtoon_policy": row["웹툰_처리정책"],
            "image_policy": row["일반이미지_처리정책"],
            "supplementary_display_condition": row["보조콘텐츠_표시조건"],
            "supplementary_link_label": row["보조콘텐츠_링크라벨"],
        })

    manifest = pd.DataFrame(records)
    if manifest["url_id"].duplicated().any():
        raise ValueError("Manifest url_id 중복")
    if len(manifest) != 42:
        raise ValueError(f"검토표는 42개여야 합니다. 현재 {len(manifest)}개")
    return manifest


# ============================================================
# HTTP 수집
# ============================================================

@dataclass
class FetchResult:
    requested_url: str
    final_url: str
    status_code: int
    ok: bool
    content_type: str
    encoding: str
    elapsed_seconds: float
    collected_at: str
    content_length: int
    raw_sha256: str
    redirect_history: list[dict[str, Any]]
    selected_headers: dict[str, str]
    body: bytes


def create_session(config: PipelineConfig) -> requests.Session:
    retry = Retry(
        total=config.max_retries,
        connect=config.max_retries,
        read=config.max_retries,
        status=config.max_retries,
        backoff_factor=0.8,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset({"GET", "POST", "HEAD"}),
        respect_retry_after_header=True,
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry, pool_connections=2, pool_maxsize=2)
    session = requests.Session()
    session.headers.update({
        "User-Agent": config.user_agent,
        "Accept-Language": "ko-KR,ko;q=0.9,en;q=0.5",
    })
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session


def choose_encoding(response: requests.Response) -> str:
    encoding = response.encoding
    if not encoding or encoding.lower() in {"iso-8859-1", "ascii"}:
        encoding = response.apparent_encoding or "utf-8"
    return encoding


def robots_allows(url: str, config: PipelineConfig) -> tuple[bool, str]:
    if not config.respect_robots_txt:
        return True, "disabled"
    parsed = urlparse(url)
    robots_url = f"{parsed.scheme}://{parsed.netloc}/robots.txt"
    try:
        parser = RobotFileParser()
        parser.set_url(robots_url)
        parser.read()
        allowed = parser.can_fetch(config.user_agent, url)
        return allowed, "allowed" if allowed else "disallowed"
    except Exception as error:
        return True, f"unavailable:{type(error).__name__}"


def response_to_fetch_result(response: requests.Response, requested_url: str, elapsed: float) -> FetchResult:
    encoding = choose_encoding(response)
    response.encoding = encoding
    selected = {
        key: value for key, value in response.headers.items()
        if key in {"Content-Type", "Content-Length", "Last-Modified", "ETag", "Cache-Control", "Content-Disposition"}
    }
    return FetchResult(
        requested_url=requested_url,
        final_url=response.url,
        status_code=response.status_code,
        ok=response.ok,
        content_type=response.headers.get("Content-Type", ""),
        encoding=encoding,
        elapsed_seconds=round(elapsed, 3),
        collected_at=now_kst_iso(),
        content_length=len(response.content),
        raw_sha256=sha256_bytes(response.content),
        redirect_history=[
            {"status_code": item.status_code, "url": item.url, "location": item.headers.get("Location")}
            for item in response.history
        ],
        selected_headers=selected,
        body=response.content,
    )


def fetch_url(session: requests.Session, url: str, config: PipelineConfig) -> FetchResult:
    allowed, status = robots_allows(url, config)
    if not allowed:
        raise PermissionError(f"robots.txt에서 허용하지 않음: {url}")
    started = time.perf_counter()
    response = session.get(url, timeout=config.request_timeout_seconds, allow_redirects=True)
    result = response_to_fetch_result(response, url, time.perf_counter() - started)
    result.selected_headers["Robots-Check"] = status
    return result


def save_fetch_result(paths: OutputPaths, row: dict[str, str], result: FetchResult) -> tuple[Path, Path]:
    folder = safe_name(row["business_function"])
    html_path = paths.raw_html / folder / f"{row['url_id']}.html"
    meta_path = paths.response_meta / folder / f"{row['url_id']}.json"
    ensure_parent(html_path)
    html_path.write_bytes(result.body)
    meta = asdict(result)
    meta.pop("body", None)
    meta.update({
        "url_id": row["url_id"],
        "business_function": row["business_function"],
        "page_title_manifest": row["page_title"],
        "fetch_strategy": row["fetch_strategy"],
        "parser_profile": row["parser_profile"],
        "raw_html_path": str(html_path.relative_to(paths.root)),
    })
    write_json(meta_path, meta)
    return html_path, meta_path


# ============================================================
# 결정론적 HTML 파싱
# ============================================================

def safe_text(node: Tag) -> str:
    fragment = BeautifulSoup(str(node), "lxml")
    root = fragment.body.find() if fragment.body else fragment.find()
    if not root:
        return ""
    for child in root.select(
        ".sr-only,.hide,script,style,noscript,template,.floatTop,.floatBottom,.paging,.pagination"
    ):
        child.decompose()
    for image in root.find_all("img"):
        image.replace_with(norm(image.get("alt", "")))
    for br in root.find_all("br"):
        br.replace_with(" ")
    return clean_visible_text(root.get_text(" ", strip=True))


def infer_file_format(node: Tag) -> str:
    text = (" ".join(node.get("class", [])).lower() + " " + norm(node.get_text(" ", strip=True)).lower())
    for keyword, fmt in [
        ("icohwp", "HWP"), ("hwp", "HWP"), ("icopdf", "PDF"), ("pdf", "PDF"),
        ("xlsx", "XLSX"), ("excel", "XLSX"), ("docx", "DOCX"), ("doc", "DOC"),
    ]:
        if keyword in text:
            return fmt
    return "FILE"


def cell_text(cell: Tag) -> str:
    fragment = BeautifulSoup(str(cell), "lxml")
    root = fragment.body.find() if fragment.body else fragment.find()
    if not root:
        return ""
    for child in root.select("script,style,noscript,template,.sr-only"):
        child.decompose()
    for button in root.find_all("button"):
        fmt = infer_file_format(button)
        label = re.sub(r"다운로드$", "", norm(button.get_text(" ", strip=True))).strip()
        button.replace_with(f"{label} {fmt} 다운로드".strip())
    for image in root.find_all("img"):
        image.replace_with(norm(image.get("alt", "")))
    for br in root.find_all("br"):
        br.replace_with(" ")
    return norm(root.get_text(" ", strip=True))


def expand_table(table: Tag) -> tuple[list[str], list[list[str]]]:
    grid: list[list[str]] = []
    active: dict[int, list[Any]] = {}
    header_flags: list[bool] = []
    max_columns = 0

    for tr in table.find_all("tr"):
        row: list[str] = []
        column = 0
        cells = tr.find_all(["th", "td"], recursive=False)

        def fill_active() -> None:
            nonlocal column
            while column in active:
                remaining, value = active[column]
                while len(row) <= column:
                    row.append("")
                row[column] = value
                remaining -= 1
                if remaining <= 0:
                    del active[column]
                else:
                    active[column] = [remaining, value]
                column += 1

        fill_active()
        header_flags.append(any(cell.name == "th" for cell in cells))
        for cell in cells:
            fill_active()
            text = cell_text(cell)
            rowspan = max(1, int(cell.get("rowspan", 1) or 1))
            colspan = max(1, int(cell.get("colspan", 1) or 1))
            for offset in range(colspan):
                target = column + offset
                while len(row) <= target:
                    row.append("")
                row[target] = text
                if rowspan > 1:
                    active[target] = [rowspan - 1, text]
            column += colspan
        if active:
            final_col = max(active)
            while column <= final_col:
                if column in active:
                    fill_active()
                else:
                    while len(row) <= column:
                        row.append("")
                    column += 1
        max_columns = max(max_columns, len(row))
        if row and any(row):
            grid.append(row)

    if not grid:
        return [], []
    for row in grid:
        row.extend([""] * (max_columns - len(row)))
    if table.thead:
        header_count = len(table.thead.find_all("tr", recursive=False))
    else:
        header_count = 1 if header_flags and header_flags[0] else 0
    header_count = max(1, header_count)
    header_rows = grid[:header_count]
    rows = grid[header_count:]
    headers: list[str] = []
    for col in range(max_columns):
        values: list[str] = []
        for hrow in header_rows:
            value = norm(hrow[col])
            if value and value not in values:
                values.append(value)
        headers.append(" / ".join(values) if values else f"열{col + 1}")
    counts: dict[str, int] = {}
    unique_headers: list[str] = []
    for header in headers:
        counts[header] = counts.get(header, 0) + 1
        unique_headers.append(header if counts[header] == 1 else f"{header} {counts[header]}")
    return unique_headers, rows




def classify_action_type(label: str, expected_types: str = "") -> str:
    text = norm(label)
    tests = [
        ("자가진단", "self_check"), ("진행상황", "progress"), ("신고", "report"),
        ("상담", "consult"), ("부채증명", "issue"), ("발급", "issue"),
        ("환급", "refund"), ("법령", "legal_reference"), ("규정", "legal_reference"),
        ("위치", "location"), ("찾아오시는", "location"), ("전화", "contact"),
        ("문의", "contact"), ("조회", "lookup"), ("다운로드", "download"),
        ("신청", "apply"), ("접수", "apply"),
    ]
    for keyword, action_type in tests:
        if keyword in text:
            return action_type
    return "related_service"


def allowed_action_domain(url: str) -> bool:
    if url.startswith(("tel:", "mailto:")):
        return True
    host = (urlparse(url).hostname or "").lower()
    return any(host == suffix or host.endswith("." + suffix) for suffix in ALLOWED_ACTION_DOMAIN_SUFFIXES)


def _extract_js_url(onclick: str) -> str | None:
    for pattern in URL_IN_JS_PATTERNS:
        match = pattern.search(onclick or "")
        if match:
            return match.group(1)
    direct = re.search(r"https?://[^'\"\s)]+", onclick or "")
    return direct.group(0) if direct else None


def action_label_with_context(node: Tag, label: str) -> str:
    label = norm(label)
    if label not in {"바로가기", "신청", "조회", "다운로드"}:
        return label
    parent = node.find_parent(["div", "li", "td", "section", "article"])
    if parent:
        context_node = parent.find(["strong", "h2", "h3", "h4", "dt"], recursive=True)
        context = norm(context_node.get_text(" ", strip=True)) if context_node else ""
        if context and context != label:
            return f"{context} {label}"
    previous = node.find_previous(["strong", "h2", "h3", "h4", "dt"])
    context = norm(previous.get_text(" ", strip=True)) if previous else ""
    return f"{context} {label}".strip() if context else label


def extract_actions(
    root: Tag,
    base_url: str,
    manifest_row: dict[str, str],
    attachments: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    actions: list[dict[str, Any]] = []
    expected = manifest_row.get("expected_action_types", "")
    expected_set = {value.strip() for value in expected.split("|") if value.strip()}
    action_policy = manifest_row.get("action_policy", "")
    if action_policy.startswith("X"):
        return []
    seen: set[tuple[str, str]] = set()

    def add(label: str, url: str, source_tag: str, js_function: str = "") -> None:
        label = norm(label)
        if not label or is_noise_text(label):
            return
        if any(noise in label for noise in ["디지털역사관", "예솜이", "챗봇", "앱 설치", "공식 홈페이지"]):
            return
        action_like = any(keyword in label for keyword in ACTION_KEYWORDS)
        if not action_like:
            return
        action_type = classify_action_type(label, expected)
        host = (urlparse(url).hostname or "").lower()
        if host == "law.go.kr" or host.endswith(".law.go.kr"):
            action_type = "legal_reference"
        if expected_set and action_type not in expected_set and action_type not in {"download", "legal_reference", "location", "contact"}:
            return
        key = (url, label)
        if key in seen:
            return
        seen.add(key)
        actions.append({
            "action_id": sha256_text(f"{manifest_row['url_id']}|{url}|{label}")[:16],
            "label": label,
            "url": url,
            "action_type": action_type,
            "source_tag": source_tag,
            "javascript_function": js_function,
            "auth_required": manifest_row.get("action_auth", manifest_row.get("auth_required", "")),
            "official_domain": allowed_action_domain(url),
            "requires_review": not allowed_action_domain(url),
        })

    for anchor in root.find_all("a", href=True):
        href = anchor.get("href", "").strip()
        url = absolute_url(base_url, href, allow_contact=True)
        if not url:
            continue
        label = norm(anchor.get_text(" ", strip=True)) or norm(anchor.get("title", ""))
        label = action_label_with_context(anchor, label)
        if any(keyword in label for keyword in ACTION_KEYWORDS):
            add(label, url, "a")

    for node in root.find_all(["button", "a", "input"], onclick=True):
        onclick = node.get("onclick", "")
        candidate = _extract_js_url(onclick)
        if not candidate:
            continue
        url = absolute_url(base_url, candidate, allow_contact=True)
        if not url:
            continue
        label = norm(node.get_text(" ", strip=True)) or norm(node.get("value", "")) or norm(node.get("title", ""))
        label = action_label_with_context(node, label)
        function_name = onclick.split("(", 1)[0].strip()
        add(label, url, node.name, function_name)

    # 다운로드도 챗봇 Action으로 제공할 수 있도록 연결합니다.
    for attachment in attachments:
        url = attachment.get("url") or base_url
        label = attachment.get("display_name", "첨부파일 다운로드")
        key = (url, label)
        if key in seen:
            continue
        seen.add(key)
        actions.append({
            "action_id": attachment["attachment_id"],
            "label": label,
            "url": url,
            "action_type": "download",
            "source_tag": "attachment",
            "attachment_id": attachment["attachment_id"],
            "download_method": attachment["download_method"],
            "auth_required": "불필요",
            "official_domain": True,
            "requires_review": False,
        })

    return actions


def extract_links(root: Tag, base_url: str) -> list[dict[str, Any]]:
    links: list[dict[str, Any]] = []
    seen: set[tuple[str, str]] = set()
    for anchor in root.find_all("a", href=True):
        url = absolute_url(base_url, anchor.get("href"))
        text = norm(anchor.get_text(" ", strip=True))
        if not url or not text or is_noise_text(text):
            continue
        key = (url, text)
        if key in seen:
            continue
        seen.add(key)
        links.append({"url": url, "anchor_text": text})
    return links


def _looks_decorative_image(image: Tag, filename: str, alt: str) -> bool:
    lowered = " ".join([
        filename.lower(), alt.lower(),
        " ".join(image.get("class", [])).lower(),
        norm(image.get("id", "")).lower(),
    ])
    decorative_tokens = {
        "logo", "icon", "ico_", "bullet", "btn_", "button", "banner",
        "bg_", "background", "qr", "character", "mascot", "예솜이",
    }
    return any(token in lowered for token in decorative_tokens)


def extract_videos(root: Tag, base_url: str, manifest_row: dict[str, str]) -> list[dict[str, Any]]:
    """영상 URL은 관리용 메타데이터로만 저장하고 다운로드·인덱싱하지 않습니다."""
    videos: list[dict[str, Any]] = []
    seen: set[str] = set()

    def add(media_url: str | None, poster_url: str | None, source_tag: str, label: str) -> None:
        url = absolute_url(base_url, media_url)
        if not url or url in seen:
            return
        seen.add(url)
        videos.append({
            "video_id": sha256_text(f"{manifest_row['url_id']}|{url}")[:16],
            "label": label or f"{manifest_row['page_title']} 영상",
            "media_url": url,
            "poster_url": absolute_url(base_url, poster_url),
            "source_page_url": manifest_row["source_url"],
            "source_tag": source_tag,
            "indexable": False,
            "download": False,
            "content_role": "supplementary",
            "delivery_mode": "official_page",
        })

    for video in root.find_all("video"):
        label = norm(video.get("title", "")) or norm(video.get("aria-label", ""))
        add(video.get("src"), video.get("poster"), "video", label)
        for source_node in video.find_all("source", src=True):
            add(source_node.get("src"), video.get("poster"), "video/source", label)

    for iframe in root.find_all("iframe", src=True):
        src = iframe.get("src", "")
        lowered = src.lower()
        if any(token in lowered for token in ["youtube", "youtu.be", "vimeo", ".mp4", ".webm"]):
            label = norm(iframe.get("title", ""))
            add(src, None, "iframe", label)

    return videos


def _has_webtoon(root: Tag) -> bool:
    for image in root.find_all("img"):
        value = " ".join([
            image.get("src", ""), image.get("data-src", ""),
            image.get("alt", ""), image.get("title", ""),
        ]).lower()
        if "webtoon" in value or "웹툰" in value:
            return True
    return "웹툰" in norm(root.get_text(" ", strip=True))


def extract_supplementary_links(
    root: Tag,
    base_url: str,
    manifest_row: dict[str, str],
    videos: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    """영상·웹툰은 Action이 아닌 참고 링크로 공식 게시 페이지만 제공합니다."""
    links: list[dict[str, Any]] = []
    label_override = norm(manifest_row.get("supplementary_link_label", ""))
    display_condition = norm(manifest_row.get("supplementary_display_condition", ""))

    if videos and manifest_row.get("video_policy") == "reference_only_official_page":
        links.append({
            "supplementary_id": sha256_text(f"{manifest_row['url_id']}|video|{manifest_row['source_url']}")[:16],
            "label": label_override or f"{manifest_row['page_title']} 영상 보기",
            "url": manifest_row["source_url"],
            "link_type": "video",
            "indexable": False,
            "content_status": "reference_only",
            "display_condition": display_condition or "user_requests_video",
            "media_urls": [item["media_url"] for item in videos],
            "notice": "영상은 이해를 돕는 참고 자료이며 최신 제도 답변의 직접 근거로 사용하지 않습니다.",
        })

    if _has_webtoon(root) and manifest_row.get("webtoon_policy") == "reference_only_official_page":
        links.append({
            "supplementary_id": sha256_text(f"{manifest_row['url_id']}|webtoon|{manifest_row['source_url']}")[:16],
            "label": label_override or f"{manifest_row['page_title']} 웹툰 보기",
            "url": manifest_row["source_url"],
            "link_type": "webtoon",
            "indexable": False,
            "content_status": "reference_only",
            "display_condition": display_condition or "user_requests_visual_guide",
            "notice": "웹툰은 참고용 시각 자료이며 금액·신청 조건은 현재 공식 본문을 우선합니다.",
        })

    return links


def extract_images(
    root: Tag,
    base_url: str,
    manifest_row: dict[str, str],
    video_poster_urls: set[str] | None = None,
) -> list[dict[str, Any]]:
    """웹툰·영상 포스터·장식 이미지는 제외하고 핵심 이미지 메타데이터만 저장합니다."""
    images: list[dict[str, Any]] = []
    seen: set[str] = set()
    poster_urls = video_poster_urls or set()
    for image in root.find_all("img"):
        source = image.get("src") or image.get("data-src") or image.get("data-original")
        url = absolute_url(base_url, source)
        if not url or url in seen or url in poster_urls:
            continue
        alt = norm(image.get("alt", ""))
        filename = Path(urlparse(url).path).name
        lowered = f"{filename} {alt}".lower()

        # 웹툰은 이미지 목록·다운로드·RAG에서 제외하고 공식 페이지 참고 링크만 제공합니다.
        if "webtoon" in lowered or "웹툰" in lowered:
            continue
        if _looks_decorative_image(image, filename, alt):
            continue

        role = "content_image"
        if any(keyword in lowered for keyword in ["proc", "process", "step", "flow", "절차", "과정"]):
            role = "process_diagram"
        elif not alt:
            # 파일명만으로 의미를 판단할 수 없는 이미지는 저장하지 않습니다.
            continue

        seen.add(url)
        images.append({
            "url": url,
            "alt": alt,
            "filename": filename,
            "image_role": role,
            "indexable": False,
            "download_policy": manifest_row.get("image_policy", "metadata_only_nondecorative"),
        })
    return images


class StructureParser:
    def __init__(self, page_type: str = "static_page"):
        self.blocks: list[dict[str, Any]] = []
        self.page_type = page_type

    def add(self, block: dict[str, Any] | None) -> None:
        if not block:
            return
        if self.blocks and block.get("type") in {"paragraph", "heading"}:
            if self.blocks[-1].get("type") == block.get("type") and self.blocks[-1].get("text") == block.get("text"):
                return
        self.blocks.append(block)

    def parse(self, root: Tag) -> list[dict[str, Any]]:
        faq_dls = root.select(".accoWrap .accodian > dl, .accodian.con.ico > dl, .accordion > dl")
        if self.page_type == "faq" or faq_dls:
            if not faq_dls:
                faq_dls = [dl for dl in root.find_all("dl") if dl.find("dt", recursive=False) and dl.find("dd", recursive=False)]
            faq_ids = {id(node) for node in faq_dls}
            for child in root.find_all(recursive=False):
                if isinstance(child, Tag) and any(id(dl) in faq_ids for dl in child.select("dl")):
                    continue
                self.process_node(child)
            for dl in faq_dls:
                self.parse_faq(dl)
        else:
            for child in root.find_all(recursive=False):
                self.process_node(child)
        return self.blocks

    def parse_faq(self, dl: Tag) -> None:
        dt = dl.find("dt", recursive=False)
        dd = dl.find("dd", recursive=False)
        if not dt or not dd:
            return
        question = re.sub(r"^\s*질문\s*", "", safe_text(dt))
        question = re.sub(r"\s*열기\s*$", "", question)
        answer_parser = StructureParser("static_page")
        answer_parser.process_node(dd)
        if not answer_parser.blocks:
            answer_parser.add({"type": "paragraph", "text": re.sub(r"^\s*답변\s*", "", safe_text(dd))})
        answer_text = blocks_text(answer_parser.blocks)
        if question and answer_text:
            self.add({
                "type": "faq", "question": norm(question),
                "answer_blocks": answer_parser.blocks, "answer_text": answer_text,
            })

    def parse_definition_list(self, dl: Tag) -> None:
        children = [child for child in dl.find_all(recursive=False) if isinstance(child, Tag)]
        index = 0
        while index < len(children):
            if children[index].name != "dt":
                self.process_node(children[index])
                index += 1
                continue
            dt = children[index]
            dd = children[index + 1] if index + 1 < len(children) and children[index + 1].name == "dd" else None
            term = re.sub(r"\s*열기\s*$", "", safe_text(dt))
            term = re.sub(r"^\s*\d{1,2}\s+(?=\D)", "", term)
            if dd:
                parser = StructureParser("static_page")
                parser.process_node(dd)
                if not parser.blocks:
                    parser.add({"type": "paragraph", "text": safe_text(dd)})
                self.add({
                    "type": "definition", "term": norm(term),
                    "definition_blocks": parser.blocks,
                    "definition_text": blocks_text(parser.blocks),
                })
                index += 2
            else:
                self.add({"type": "heading", "level": 3, "text": norm(term)})
                index += 1

    def process_node(self, node: Any) -> None:
        if isinstance(node, Comment):
            return
        if isinstance(node, NavigableString):
            text = norm(str(node))
            if text and not is_noise_text(text):
                self.add({"type": "paragraph", "text": text})
            return
        if not isinstance(node, Tag):
            return
        name = node.name.lower()
        classes = set(node.get("class", []))
        if name in {"script", "style", "noscript", "template"} or classes & {"floatTop", "floatBottom", "paging", "pagination", "sr-only"}:
            return
        if name in {"button", "input"}:
            return
        if name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
            text = safe_text(node)
            if text and not is_noise_text(text):
                self.add({"type": "heading", "level": int(name[1]), "text": text})
            return
        if name == "p":
            text = safe_text(node)
            if text and not is_noise_text(text):
                self.add({"type": "paragraph", "text": text})
            return
        if name == "table":
            headers, rows = expand_table(node)
            if headers or rows:
                self.add({"type": "table", "headers": headers, "rows": rows, "row_count": len(rows)})
            return
        if name in {"ul", "ol"}:
            items: list[str] = []
            for li in node.find_all("li", recursive=False):
                fragment = BeautifulSoup(str(li), "lxml")
                copied = fragment.body.find("li") if fragment.body else None
                if not copied:
                    continue
                for nested in copied.find_all(["ul", "ol"]):
                    nested.decompose()
                text = safe_text(copied)
                if text and not is_noise_text(text):
                    items.append(text)
            # 단순 탭 메뉴는 제거합니다.
            if set(items).issubset({"착오송금인", "착오송금수취인"}):
                items = []
            if items:
                self.add({"type": "list", "ordered": name == "ol", "items": items})
            for li in node.find_all("li", recursive=False):
                for nested in li.find_all(["ul", "ol"], recursive=False):
                    self.process_node(nested)
            return
        if name == "dl":
            self.parse_definition_list(node)
            return
        if name == "figure":
            caption = node.find("figcaption")
            if caption:
                text = safe_text(caption)
                if text and not is_noise_text(text):
                    self.add({"type": "paragraph", "text": text})
            return

        inline: list[str] = []
        def flush() -> None:
            nonlocal inline
            text = norm(" ".join(inline))
            inline = []
            if text and not is_noise_text(text):
                self.add({"type": "paragraph", "text": text})

        for child in node.children:
            if isinstance(child, Comment):
                continue
            if isinstance(child, NavigableString):
                value = norm(str(child))
                if value:
                    inline.append(value)
            elif isinstance(child, Tag):
                if child.name.lower() == "br":
                    inline.append(" ")
                elif child.name.lower() in BLOCK_TAGS:
                    flush()
                    self.process_node(child)
                elif child.name.lower() in {"button", "input"}:
                    continue
                else:
                    value = safe_text(child)
                    if value:
                        inline.append(value)
        flush()


def render_blocks(blocks: list[dict[str, Any]]) -> str:
    lines: list[str] = []
    for block in blocks:
        kind = block.get("type")
        if kind == "heading":
            level = min(max(int(block.get("level", 2)), 2), 6)
            lines.append(f"{'#' * level} {block.get('text', '')}")
        elif kind == "paragraph":
            lines.append(block.get("text", ""))
        elif kind == "list":
            for index, item in enumerate(block.get("items", []), start=1):
                prefix = f"{index}. " if block.get("ordered") else "- "
                lines.append(prefix + item)
        elif kind == "definition":
            lines.append("### " + block.get("term", ""))
            lines.append(render_blocks(block.get("definition_blocks", [])))
        elif kind == "faq":
            lines.append("### Q. " + block.get("question", ""))
            lines.append(render_blocks(block.get("answer_blocks", [])))
        elif kind == "table":
            headers = [norm(value).replace("|", "\\|") for value in block.get("headers", [])]
            if headers:
                lines.append("| " + " | ".join(headers) + " |")
                lines.append("| " + " | ".join(["---"] * len(headers)) + " |")
                for row in block.get("rows", []):
                    values = [norm(value).replace("|", "\\|") for value in row]
                    values.extend([""] * (len(headers) - len(values)))
                    lines.append("| " + " | ".join(values[:len(headers)]) + " |")
        lines.append("")
    return re.sub(r"\n{3,}", "\n\n", "\n".join(lines)).strip()


def blocks_text(blocks: list[dict[str, Any]]) -> str:
    values: list[str] = []
    for block in blocks:
        kind = block.get("type")
        if kind in {"heading", "paragraph"}:
            values.append(block.get("text", ""))
        elif kind == "list":
            values.extend(block.get("items", []))
        elif kind == "definition":
            values.extend([block.get("term", ""), block.get("definition_text", "")])
        elif kind == "faq":
            values.extend([block.get("question", ""), block.get("answer_text", "")])
        elif kind == "table":
            values.extend(block.get("headers", []))
            for row in block.get("rows", []):
                values.extend(row)
    return norm(" ".join(values))


def iter_blocks_recursive(blocks: list[dict[str, Any]]) -> Iterable[dict[str, Any]]:
    for block in blocks:
        yield block
        for key in ("definition_blocks", "answer_blocks"):
            children = block.get(key, [])
            if isinstance(children, list):
                yield from iter_blocks_recursive(children)


def count_block_type(blocks: list[dict[str, Any]], target: str) -> int:
    return sum(1 for block in iter_blocks_recursive(blocks) if block.get("type") == target)


def block_fingerprint(block: dict[str, Any]) -> str:
    return sha256_text(json.dumps(block, ensure_ascii=False, sort_keys=True))


def apply_document_filters(document: dict[str, Any]) -> None:
    excluded: list[dict[str, Any]] = []
    cleaned: list[dict[str, Any]] = []
    for block in document.get("blocks", []):
        text = blocks_text([block])
        if document["doc_id"] == "MT-009" and any(token in text for token in ["5천만 원", "5천만원", "1천만원 이하", "1천만원 초과"]):
            excluded.append({"reason": "구버전 지원금액이 포함된 웹툰 설명", "content": text})
            continue
        if is_noise_text(text):
            continue
        cleaned.append(block)
    document["blocks"] = cleaned
    if excluded:
        document["excluded_content"] = excluded
        document["content_filter"] = "legacy_amount_removed"


def refresh_document(document: dict[str, Any]) -> None:
    document["content_markdown"] = render_blocks(document.get("blocks", []))
    document["content_text"] = blocks_text(document.get("blocks", []))
    parsed_hash = sha256_text(document["content_markdown"])
    document["parsed_content_sha256"] = parsed_hash
    document["content_hash"] = parsed_hash


def select_content_root(soup: BeautifulSoup) -> Tag:
    root = (
        soup.select_one(".contents") or soup.select_one("#contents") or
        soup.select_one("#content") or soup.select_one("main") or soup.body
    )
    if not root:
        raise ValueError("본문 루트를 찾지 못했습니다.")
    return root


def parse_html_document(
    html_bytes: bytes,
    final_url: str,
    manifest_row: dict[str, str],
    encoding: str,
) -> dict[str, Any]:
    soup = BeautifulSoup(html_bytes.decode(encoding, errors="replace"), "lxml")
    root = select_content_root(soup)

    attachments = extract_attachments(root, final_url)
    actions = extract_actions(root, final_url, manifest_row, attachments)
    if not actions and manifest_row.get("page_type") == "dynamic_lookup":
        actions = [{
            "action_id": sha256_text(f"{manifest_row['url_id']}|{manifest_row['source_url']}|lookup")[:16],
            "label": f"{manifest_row['page_title']} 조회 바로가기",
            "url": manifest_row["source_url"],
            "action_type": "lookup",
            "source_tag": "source_page",
            "auth_required": manifest_row.get("action_auth", "불필요"),
            "official_domain": True,
            "requires_review": False,
        }]
    videos = extract_videos(root, final_url, manifest_row)
    supplementary_links = extract_supplementary_links(root, final_url, manifest_row, videos)
    poster_urls = {item.get("poster_url") for item in videos if item.get("poster_url")}
    images = extract_images(root, final_url, manifest_row, poster_urls)
    links = extract_links(root, final_url)

    source_fragment = BeautifulSoup(str(root), "lxml")
    source_root = source_fragment.body.find() if source_fragment.body else source_fragment.find()
    if source_root:
        for selector in NOISE_SELECTORS:
            for node in source_root.select(selector):
                node.decompose()
        source_text = norm(source_root.get_text(" ", strip=True))
    else:
        source_text = ""

    for selector in NOISE_SELECTORS:
        for node in root.select(selector):
            node.decompose()
    for media_node in root.find_all(["video", "source", "iframe", "embed", "object"]):
        media_node.decompose()
    # 웹툰 이미지는 본문 텍스트·청킹 대상에서도 제거합니다.
    for image_node in list(root.find_all("img")):
        media_value = " ".join([
            image_node.get("src", ""), image_node.get("data-src", ""),
            image_node.get("alt", ""), image_node.get("title", ""),
        ]).lower()
        if "webtoon" in media_value or "웹툰" in media_value:
            image_node.decompose()

    parser = StructureParser(manifest_row.get("page_type", "static_page"))
    blocks = parser.parse(root)
    document = {
        "doc_id": manifest_row["url_id"],
        "parent_doc_id": manifest_row["url_id"],
        "record_type": "page",
        "indexable": manifest_row.get("rag_index_mode") != "none",
        "rag_index_mode": manifest_row.get("rag_index_mode", "full"),
        "title": manifest_row["page_title"],
        "manifest_title": manifest_row["page_title"],
        "html_title": norm(soup.title.get_text(" ", strip=True)) if soup.title else "",
        "source_url": manifest_row["source_url"],
        "final_url": final_url,
        "site_name": manifest_row["site_name"],
        "business_function": manifest_row["business_function"],
        "target_business_function": manifest_row.get("target_business_function", manifest_row["business_function"]),
        "page_type": manifest_row["page_type"],
        "parser_profile": manifest_row["parser_profile"],
        "breadcrumb": [manifest_row["business_function"], manifest_row["page_title"]],
        "blocks": blocks,
        "links": links,
        "actions": actions,
        "attachments": attachments,
        "images": images,
        "videos": videos,
        "supplementary_links": supplementary_links,
        "policy": {
            "normalized_decision": manifest_row["normalized_decision"],
            "content_scope": manifest_row["content_scope"],
            "rag_index_label": manifest_row["rag_index_label"],
            "action_policy": manifest_row["action_policy"],
            "expected_action_types": manifest_row["expected_action_types"],
            "pagination_policy": manifest_row["pagination_policy"],
            "auth_required": manifest_row["auth_required"],
            "attachment_download_policy": manifest_row["attachment_download_policy"],
            "attachment_rag_policy": manifest_row["attachment_rag_policy"],
            "attachment_user_delivery_policy": manifest_row["attachment_user_delivery_policy"],
            "video_policy": manifest_row["video_policy"],
            "webtoon_policy": manifest_row["webtoon_policy"],
            "image_policy": manifest_row["image_policy"],
        },
        "parsed_at": now_kst_iso(),
    }
    apply_document_filters(document)
    refresh_document(document)
    document["quality"] = build_quality(document, source_text, manifest_row)
    return document


def build_quality(document: dict[str, Any], source_text: str, row: dict[str, str]) -> dict[str, Any]:
    blocks = document.get("blocks", [])
    content = document.get("content_text", "")
    faq_count = count_block_type(blocks, "faq")
    table_count = count_block_type(blocks, "table")
    markdown_lines = {norm(line.lstrip("#-0123456789. ")) for line in document.get("content_markdown", "").splitlines()}
    noise_hits = [value for value in NOISE_EXACT_TEXTS if value in markdown_lines]
    reasons: list[str] = []
    if document.get("indexable", True) and len(content) < 80:
        reasons.append("본문이 80자 미만")
    retention = round(len(content) / max(1, len(source_text)), 3)
    if document.get("indexable", True) and retention < 0.55:
        reasons.append("본문 보존율 55% 미만")
    if noise_hits:
        reasons.append("공통 UI 문구 잔존")
    if row.get("page_type") == "faq" and faq_count == 0:
        reasons.append("FAQ 질문-답변 추출 실패")
    if row.get("action_policy", "").startswith("O") and not document.get("actions"):
        reasons.append("예상 Action 링크 미검출")
    status = "fail" if any(value in reasons for value in ["공통 UI 문구 잔존", "FAQ 질문-답변 추출 실패"]) else ("review" if reasons else "pass")
    return {
        "status": status,
        "reasons": reasons,
        "source_text_chars": len(source_text),
        "parsed_text_chars": len(content),
        "retention_ratio": retention,
        "faq_count": faq_count,
        "table_count": table_count,
        "attachment_button_count": len(document.get("attachments", [])),
        "action_count": len(document.get("actions", [])),
        "video_count": len(document.get("videos", [])),
        "supplementary_link_count": len(document.get("supplementary_links", [])),
        "noise_hits": noise_hits,
    }


# ============================================================
# 공통 페이지네이션 수집
# ============================================================

@dataclass
class PaginationPlan:
    method: str
    endpoint: str
    page_param: str
    page_size_param: str | None
    page_size: int
    last_internal_index: int
    index_base: int
    base_payload: dict[str, str]
    detection_method: str


def extract_form_payload(form: Tag | None) -> dict[str, str]:
    payload: dict[str, str] = {}
    if not form:
        return payload
    for field in form.select("input[name], select[name], textarea[name]"):
        name = norm(field.get("name", ""))
        if not name:
            continue
        if field.name == "select":
            selected = field.find("option", selected=True) or field.find("option")
            value = selected.get("value", "") if selected else ""
        elif field.name == "textarea":
            value = field.get_text("", strip=False)
        else:
            field_type = norm(field.get("type", "text")).lower()
            if field_type in {"checkbox", "radio"} and not field.has_attr("checked"):
                continue
            value = field.get("value", "")
        payload[name] = str(value or "")
    return payload


def extract_script_text(soup: BeautifulSoup) -> str:
    return "\n".join(script.get_text("\n", strip=True) for script in soup.find_all("script") if not script.get("src"))


def detect_pagination_plan(
    html_bytes: bytes,
    encoding: str,
    final_url: str,
    row: dict[str, str],
    config: PipelineConfig,
) -> PaginationPlan | None:
    if not config.enable_generic_pagination:
        return None
    policy = row.get("pagination_policy", "")
    if "제외" in policy:
        return None

    soup = BeautifulSoup(html_bytes.decode(encoding, errors="replace"), "lxml")
    onclick_indexes: list[int] = []
    for node in soup.select(".paging [onclick*='chgPagingNo'], .pagination [onclick*='chgPagingNo']"):
        match = re.search(r"chgPagingNo\(\s*(\d+)\s*\)", node.get("onclick", ""))
        if match:
            onclick_indexes.append(int(match.group(1)))

    if onclick_indexes:
        last_index = max(onclick_indexes)
        if last_index <= 0:
            return None
        form = soup.select_one("form#srchForm") or soup.select_one("form[name='srchForm']")
        payload = extract_form_payload(form)
        scripts = extract_script_text(soup)
        action = norm(form.get("action", "")) if form else ""
        if not action:
            match = re.search(r"[\"']#srchForm[\"']\)\.attr\(\s*[\"']action[\"']\s*,\s*[\"']([^\"']+)[\"']", scripts)
            if not match:
                match = re.search(r"attr\(\s*[\"']action[\"']\s*,\s*[\"']([^\"']+)[\"']", scripts)
            action = match.group(1) if match else ""
        endpoint = urljoin(final_url, action) if action else final_url
        page_param = "curPage"
        for candidate in ["curPage", "pageIndex", "pageNo", "currentPage", "page"]:
            if candidate in payload or re.search(rf"name\s*[:=]\s*['\"]{candidate}['\"]", scripts):
                page_param = candidate
                break
        page_size_param = "pageSize" if "pageSize" in payload or "pageSize" in scripts else None
        return PaginationPlan(
            method="POST", endpoint=endpoint, page_param=page_param,
            page_size_param=page_size_param, page_size=config.pagination_page_size,
            last_internal_index=last_index, index_base=0, base_payload=payload,
            detection_method="chgPagingNo",
        )

    # query string 기반 페이지 링크도 지원합니다.
    page_links: list[tuple[str, str, int]] = []
    for anchor in soup.select(".paging a[href], .pagination a[href]"):
        href = anchor.get("href", "")
        if not href or href.startswith("javascript:"):
            continue
        absolute = urljoin(final_url, href)
        query = parse_qs(urlparse(absolute).query)
        for param in ["curPage", "pageIndex", "pageNo", "currentPage", "page"]:
            if param in query and query[param] and str(query[param][0]).isdigit():
                page_links.append((absolute, param, int(query[param][0])))
    if page_links:
        _, param, max_page = max(page_links, key=lambda item: item[2])
        if max_page <= 1:
            return None
        return PaginationPlan(
            method="GET", endpoint=final_url, page_param=param, page_size_param=None,
            page_size=config.pagination_page_size, last_internal_index=max_page,
            index_base=1, base_payload={}, detection_method="query_link",
        )
    return None


def displayed_page_number(html_bytes: bytes, encoding: str) -> int | None:
    soup = BeautifulSoup(html_bytes.decode(encoding, errors="replace"), "lxml")
    current = soup.select_one(".paging strong[title*='현재 페이지'] span") or soup.select_one(".pagination .active")
    if not current:
        return None
    text = norm(current.get_text(" ", strip=True))
    return int(text) if text.isdigit() else None


def repeatable_signature(document: dict[str, Any]) -> str:
    selected = [
        block for block in document.get("blocks", [])
        if block.get("type") in {"faq", "table", "list", "definition"}
    ]
    if not selected:
        selected = document.get("blocks", [])
    return sha256_text(json.dumps(selected, ensure_ascii=False, sort_keys=True))


def merge_table_blocks(target: dict[str, Any], incoming: dict[str, Any]) -> None:
    existing = {tuple(row) for row in target.get("rows", [])}
    for row in incoming.get("rows", []):
        key = tuple(row)
        if key not in existing:
            existing.add(key)
            target.setdefault("rows", []).append(row)
    target["row_count"] = len(target.get("rows", []))


def merge_paginated_documents(documents: list[dict[str, Any]], row: dict[str, str]) -> dict[str, Any]:
    merged = documents[0]
    page_type = row.get("page_type", "")

    if page_type == "faq":
        base_non_faq = [block for block in merged["blocks"] if block.get("type") != "faq"]
        faq_blocks: list[dict[str, Any]] = []
        seen: set[tuple[str, str]] = set()
        for document in documents:
            for block in document.get("blocks", []):
                if block.get("type") != "faq":
                    continue
                key = (norm(block.get("question", "")), norm(block.get("answer_text", "")))
                if key in seen:
                    continue
                seen.add(key)
                faq_blocks.append(block)
        merged["blocks"] = base_non_faq + faq_blocks
    else:
        # 같은 헤더를 가진 표는 행을 합치고, 나머지 새 블록은 중복 없이 추가합니다.
        table_by_headers: dict[tuple[str, ...], dict[str, Any]] = {
            tuple(block.get("headers", [])): block
            for block in merged.get("blocks", []) if block.get("type") == "table"
        }
        fingerprints = {block_fingerprint(block) for block in merged.get("blocks", [])}
        for document in documents[1:]:
            for block in document.get("blocks", []):
                if block.get("type") == "table":
                    key = tuple(block.get("headers", []))
                    if key in table_by_headers:
                        merge_table_blocks(table_by_headers[key], block)
                        continue
                fingerprint = block_fingerprint(block)
                if fingerprint not in fingerprints:
                    fingerprints.add(fingerprint)
                    merged["blocks"].append(block)

    for field, keys in [
        ("actions", ("url", "label")), ("attachments", ("attachment_id",)),
        ("images", ("url",)), ("videos", ("media_url",)),
        ("supplementary_links", ("url", "link_type")),
        ("links", ("url", "anchor_text")),
    ]:
        combined: list[dict[str, Any]] = []
        for document in documents:
            combined.extend(document.get(field, []))
        merged[field] = unique_dicts(combined, keys)

    apply_document_filters(merged)
    refresh_document(merged)
    return merged


def collect_paginated_document(
    session: requests.Session,
    first_result: FetchResult,
    first_document: dict[str, Any],
    row: dict[str, str],
    paths: OutputPaths,
    config: PipelineConfig,
) -> tuple[dict[str, Any], dict[str, Any]]:
    plan = detect_pagination_plan(first_result.body, first_result.encoding, first_result.final_url, row, config)
    if plan is None:
        collection = {
            "collection_scope": "single_public_page",
            "is_complete": True,
            "pagination_detected": False,
            "expected_page_count": 1,
            "fetched_page_count": 1,
            "failures": [],
        }
        first_document["pagination_collection"] = collection
        return first_document, collection

    if plan.last_internal_index + 1 > config.pagination_max_pages:
        raise ValueError(
            f"{row['url_id']} 페이지 수가 안전 한도 초과: {plan.last_internal_index + 1}"
        )

    page_dir = paths.pagination / row["url_id"]
    page_dir.mkdir(parents=True, exist_ok=True)
    first_page_path = page_dir / "page_001.html"
    first_page_path.write_bytes(first_result.body)

    page_documents = [first_document]
    page_records: list[dict[str, Any]] = [{
        "internal_page_index": 0 if plan.index_base == 0 else 1,
        "displayed_page_number": displayed_page_number(first_result.body, first_result.encoding),
        "status_code": first_result.status_code,
        "request_method": "GET",
        "final_url": first_result.final_url,
        "signature": repeatable_signature(first_document),
        "raw_sha256": first_result.raw_sha256,
        "raw_html_path": str(first_page_path.relative_to(paths.root)),
    }]
    failures: list[dict[str, Any]] = []
    seen_signatures = {page_records[0]["signature"]: page_records[0]["internal_page_index"]}

    if plan.index_base == 0:
        page_indexes = range(1, plan.last_internal_index + 1)
    else:
        page_indexes = range(2, plan.last_internal_index + 1)

    for page_index in page_indexes:
        try:
            started = time.perf_counter()
            if plan.method == "POST":
                payload = dict(plan.base_payload)
                payload[plan.page_param] = str(page_index)
                if plan.page_size_param:
                    payload[plan.page_size_param] = str(plan.page_size)
                response = session.post(
                    plan.endpoint, data=payload, headers={"Referer": first_result.final_url},
                    timeout=config.request_timeout_seconds, allow_redirects=True,
                )
                request_payload: dict[str, Any] = payload
            else:
                parsed = urlparse(plan.endpoint)
                query = parse_qs(parsed.query)
                query[plan.page_param] = [str(page_index)]
                query_string = urlencode(query, doseq=True)
                url = urlunparse(parsed._replace(query=query_string))
                response = session.get(url, timeout=config.request_timeout_seconds, allow_redirects=True)
                request_payload = {plan.page_param: page_index}
            response.raise_for_status()
            result = response_to_fetch_result(response, plan.endpoint, time.perf_counter() - started)
            page_document = parse_html_document(result.body, result.final_url, row, result.encoding)
            signature = repeatable_signature(page_document)
            page_number = page_index + 1 if plan.index_base == 0 else page_index
            page_path = page_dir / f"page_{page_number:03d}.html"
            page_path.write_bytes(result.body)
            record = {
                "internal_page_index": page_index,
                "displayed_page_number": displayed_page_number(result.body, result.encoding),
                "status_code": result.status_code,
                "request_method": plan.method,
                "request_payload": request_payload,
                "final_url": result.final_url,
                "signature": signature,
                "raw_sha256": result.raw_sha256,
                "raw_html_path": str(page_path.relative_to(paths.root)),
            }
            if signature in seen_signatures:
                failures.append({
                    "internal_page_index": page_index,
                    "reason": "다른 페이지와 같은 반복 콘텐츠",
                    "same_as": seen_signatures[signature],
                })
            else:
                seen_signatures[signature] = page_index
            expected_display = page_number
            actual_display = record["displayed_page_number"]
            if actual_display is not None and actual_display != expected_display:
                failures.append({
                    "internal_page_index": page_index,
                    "reason": "요청 페이지와 표시 페이지 불일치",
                    "expected": expected_display,
                    "actual": actual_display,
                })
            page_records.append(record)
            page_documents.append(page_document)
        except Exception as error:
            failures.append({
                "internal_page_index": page_index,
                "reason": "페이지 요청 또는 파싱 실패",
                "error_type": type(error).__name__,
                "error": str(error),
            })
        time.sleep(config.request_delay_seconds)

    expected_count = plan.last_internal_index + 1 if plan.index_base == 0 else plan.last_internal_index
    is_complete = not failures and len(page_documents) == expected_count
    merged = merge_paginated_documents(page_documents, row)
    collection = {
        "collection_scope": "all_public_pages" if is_complete else "partial_public_pages",
        "is_complete": is_complete,
        "pagination_detected": True,
        "detection_method": plan.detection_method,
        "method": plan.method,
        "endpoint": plan.endpoint,
        "page_param": plan.page_param,
        "index_base": plan.index_base,
        "expected_page_count": expected_count,
        "fetched_page_count": len(page_documents),
        "pages": page_records,
        "failures": failures,
        "collected_at": now_kst_iso(),
    }
    merged["pagination_collection"] = collection
    merged["dynamic_collection"] = collection if row["url_id"] == "BI-004" else None
    return merged, collection


# ============================================================
# 자산 다운로드
# ============================================================



def ensure_playwright() -> None:
    try:
        import playwright  # noqa: F401
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "playwright"], check=True)
    subprocess.run([sys.executable, "-m", "playwright", "install", "chromium"], check=True)




# ============================================================
# 문서 저장과 청킹
# ============================================================

def save_document(paths: OutputPaths, document: dict[str, Any], row: dict[str, str]) -> tuple[Path, Path]:
    folder = safe_name(row["business_function"])
    md_path = paths.parsed_markdown / folder / f"{row['url_id']}.md"
    json_path = paths.parsed_json / folder / f"{row['url_id']}.json"
    ensure_parent(md_path)
    md_path.write_text(document.get("content_markdown", ""), encoding="utf-8")
    write_json(json_path, document)
    return md_path, json_path


def render_table_chunk(headers: list[str], rows: list[list[str]]) -> str:
    return render_blocks([{"type": "table", "headers": headers, "rows": rows}])


def split_table(block: dict[str, Any], max_chars: int, heading_prefix: str = "") -> list[str]:
    headers, rows = block.get("headers", []), block.get("rows", [])
    prefix = f"### {heading_prefix}\n\n" if heading_prefix else ""
    complete = prefix + render_table_chunk(headers, rows)
    if len(complete) <= max_chars:
        return [complete]
    chunks: list[str] = []
    current: list[list[str]] = []
    for row in rows:
        candidate = prefix + render_table_chunk(headers, current + [row])
        if current and len(candidate) > max_chars:
            chunks.append(prefix + render_table_chunk(headers, current))
            current = [row]
        else:
            current.append(row)
    if current:
        chunks.append(prefix + render_table_chunk(headers, current))
    return chunks


def meaningful_chunk(text: str, min_chars: int) -> bool:
    plain = re.sub(r"[#|\-\s]", "", text)
    if len(plain) < min_chars:
        return False
    if any(fragment in text for fragment in NOISE_CONTAINS_TEXTS):
        return False
    return True




def build_link_only_document(row: dict[str, str]) -> dict[str, Any]:
    expected = [value.strip() for value in row.get("expected_action_types", "").split("|") if value.strip()]
    action_type = expected[0] if expected else "related_service"
    action = {
        "action_id": sha256_text(f"{row['url_id']}|{row['source_url']}")[:16],
        "label": row["page_title"],
        "url": row["source_url"],
        "action_type": action_type,
        "auth_required": row.get("action_auth", ""),
        "official_domain": allowed_action_domain(row["source_url"]),
        "requires_review": False,
    }
    return {
        "doc_id": row["url_id"], "parent_doc_id": row["url_id"],
        "record_type": "link_only", "indexable": False, "rag_index_mode": "none",
        "title": row["page_title"], "source_url": row["source_url"],
        "site_name": row["site_name"], "business_function": row["business_function"],
        "target_business_function": row["target_business_function"],
        "page_type": row["page_type"], "content_markdown": "", "content_text": "",
        "blocks": [], "attachments": [], "images": [], "videos": [],
        "supplementary_links": [], "links": [], "actions": [action],
        "policy": {
            "normalized_decision": row["normalized_decision"],
            "content_scope": row["content_scope"], "action_policy": row["action_policy"],
            "auth_required": row["auth_required"],
            "attachment_download_policy": row.get("attachment_download_policy", "X"),
            "attachment_rag_policy": row.get("attachment_rag_policy", "none"),
            "attachment_user_delivery_policy": row.get("attachment_user_delivery_policy", "none"),
            "video_policy": row.get("video_policy", "none"),
            "webtoon_policy": row.get("webtoon_policy", "none"),
            "image_policy": row.get("image_policy", "metadata_only_nondecorative"),
        },
        "quality": {"status": "pass", "reasons": [], "action_count": 1},
        "parsed_at": now_kst_iso(),
    }


# ============================================================
# 전체 실행
# ============================================================

# ============================================================
# 첨부파일 하이브리드 처리 및 무손실 청킹 확장
# ============================================================

import csv
import io
import struct
import xml.etree.ElementTree as ET
import zipfile
import zlib
from email.message import Message

FORM_NO_INDEX_KEYWORDS = {
    "신청서", "청구서", "지급청구서", "위임장", "동의서", "개인정보",
    "서약서", "확인서", "철회서", "신고서", "계좌신고서", "통지서",
    "양식", "서식", "작성용", "채권양도", "인감증명",
}
ATTACHMENT_INDEX_KEYWORDS = {
    "안내", "작성요령", "작성 방법", "설명", "매뉴얼", "가이드", "리플릿",
    "절차", "기준", "목록", "현황", "대상 금융회사", "구비서류 안내",
    "주요내용", "업무편람", "FAQ", "자주 묻는 질문",
}
SUPPORTED_ATTACHMENT_TEXT_FORMATS = {
    "PDF", "HWP", "HWPX", "DOCX", "XLSX", "PPTX", "CSV", "TXT",
}
OLE_MAGIC = bytes.fromhex("D0CF11E0A1B11AE1")


def extract_attachments(root: Tag, base_url: str) -> list[dict[str, Any]]:
    """다운로드 방식과 공식 링크를 함께 보존합니다."""
    attachments: list[dict[str, Any]] = []
    seen: set[Any] = set()

    for button in root.find_all(["button", "a"], onclick=True):
        match = DOWNLOAD_CALL_RE.search(button.get("onclick", ""))
        if not match:
            continue
        key = (match.group(1), match.group(2))
        if key in seen:
            continue
        seen.add(key)
        fmt = infer_file_format(button)
        label = re.sub(r"다운로드$", "", norm(button.get_text(" ", strip=True))).strip()
        row_context = ""
        parent_row = button.find_parent("tr")
        if parent_row:
            values = []
            for cell in parent_row.find_all(["th", "td"], recursive=False):
                if button in cell.descendants:
                    continue
                value = cell_text(cell)
                if value and "다운로드" not in value and value not in values:
                    values.append(value)
            row_context = " | ".join(values[:4])
        display_name = label or row_context or f"{fmt} 첨부파일"
        if fmt not in display_name.upper():
            display_name = f"{display_name} ({fmt})"
        attachments.append({
            "attachment_id": sha256_text(f"{match.group(1)}|{match.group(2)}")[:16],
            "display_name": display_name,
            "file_format": fmt,
            "download_method": "gfn_downloadFile",
            "token1": match.group(1),
            "token2": match.group(2),
            "row_context": row_context,
            "button_text": norm(button.get_text(" ", strip=True)),
            "source_page_url": base_url,
            "source_url": base_url,
            "official_download_url": None,
            "user_action_url": base_url,
            "delivery_mode": "official_download_page",
            "download_status": "metadata_only",
            "processing_policy": "metadata_only",
            "indexable": False,
            "needs_review": False,
        })

    for anchor in root.find_all("a", href=True):
        url = absolute_url(base_url, anchor.get("href"))
        if not url:
            continue
        ext = Path(urlparse(url).path).suffix.lower()
        if ext not in ATTACHMENT_EXTENSIONS:
            continue
        key = ("href", url)
        if key in seen:
            continue
        seen.add(key)
        attachments.append({
            "attachment_id": sha256_text(url)[:16],
            "display_name": norm(anchor.get_text(" ", strip=True)) or Path(urlparse(url).path).name,
            "file_format": ext.lstrip(".").upper(),
            "download_method": "href",
            "url": url,
            "source_page_url": base_url,
            "source_url": base_url,
            "official_download_url": url,
            "user_action_url": url,
            "delivery_mode": "direct_official_download",
            "download_status": "metadata_only",
            "processing_policy": "metadata_only",
            "indexable": False,
            "needs_review": False,
        })
    return attachments


def _filename_from_content_disposition(value: str) -> str | None:
    if not value:
        return None
    message = Message()
    message["content-disposition"] = value
    filename = message.get_filename()
    return filename.strip() if filename else None


def detect_attachment_format(path: Path, declared: str = "") -> str:
    data = path.read_bytes()[:8192]
    declared = (declared or path.suffix.lstrip(".")).upper()
    if data.startswith(b"%PDF-"):
        return "PDF"
    if data.startswith(OLE_MAGIC):
        return "HWP" if declared == "HWP" else declared or "OLE"
    if data.startswith(b"PK\x03\x04"):
        try:
            with zipfile.ZipFile(path) as archive:
                names = set(archive.namelist())
                lower_names = {name.lower() for name in names}
                if "mimetype" in names:
                    mime = archive.read("mimetype").decode("utf-8", errors="ignore")
                    if "hwp+zip" in mime:
                        return "HWPX"
                if any(name.startswith("word/") for name in lower_names):
                    return "DOCX"
                if any(name.startswith("xl/") for name in lower_names):
                    return "XLSX"
                if any(name.startswith("ppt/") for name in lower_names):
                    return "PPTX"
                if any(name.startswith("contents/section") for name in lower_names):
                    return "HWPX"
        except Exception:
            pass
        return declared or "ZIP"
    if declared in {"CSV", "TXT"}:
        return declared
    return declared or "UNKNOWN"


def inspect_downloaded_attachment(path: Path, declared: str = "") -> dict[str, Any]:
    data = path.read_bytes()[:8192]
    lower = data.lstrip().lower()
    html_like = lower.startswith((b"<!doctype html", b"<html", b"<script"))
    detected = detect_attachment_format(path, declared)
    return {
        "detected_format": detected,
        "signature_valid": not html_like and path.stat().st_size > 0,
        "html_error_response": html_like,
        "size_bytes": path.stat().st_size,
        "sha256": sha256_bytes(path.read_bytes()),
    }


def _clean_extracted_text(text: str, max_chars: int) -> str:
    text = text.replace("\x00", " ")
    text = re.sub(r"[\u0000-\u0008\u000b\u000c\u000e-\u001f]", " ", text)
    lines = []
    for line in text.splitlines():
        value = norm(line)
        if value and (not lines or value != lines[-1]):
            lines.append(value)
    return "\n".join(lines)[:max_chars].strip()


def extract_pdf_text(path: Path, config: PipelineConfig) -> str:
    from pypdf import PdfReader
    reader = PdfReader(str(path))
    texts = []
    for page in reader.pages[: config.attachment_pdf_max_pages]:
        texts.append(page.extract_text() or "")
    return _clean_extracted_text("\n".join(texts), config.attachment_max_text_chars)


def extract_docx_or_pptx_text(path: Path, config: PipelineConfig) -> str:
    values: list[str] = []
    with zipfile.ZipFile(path) as archive:
        targets = [
            name for name in archive.namelist()
            if (
                name.startswith("word/") and name.endswith(".xml")
            ) or (
                name.startswith("ppt/slides/") and name.endswith(".xml")
            )
        ]
        for name in sorted(targets):
            try:
                root = ET.fromstring(archive.read(name))
                text = " ".join(value.strip() for value in root.itertext() if value.strip())
                if text:
                    values.append(text)
            except Exception:
                continue
    return _clean_extracted_text("\n".join(values), config.attachment_max_text_chars)


def extract_hwpx_text(path: Path, config: PipelineConfig) -> str:
    values: list[str] = []
    with zipfile.ZipFile(path) as archive:
        targets = [
            name for name in archive.namelist()
            if name.lower().startswith("contents/section") and name.lower().endswith(".xml")
        ]
        for name in sorted(targets):
            try:
                root = ET.fromstring(archive.read(name))
                text = " ".join(value.strip() for value in root.itertext() if value.strip())
                if text:
                    values.append(text)
            except Exception:
                continue
    return _clean_extracted_text("\n".join(values), config.attachment_max_text_chars)


def extract_xlsx_text(path: Path, config: PipelineConfig) -> str:
    ns = {"m": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}
    values: list[str] = []
    with zipfile.ZipFile(path) as archive:
        shared: list[str] = []
        if "xl/sharedStrings.xml" in archive.namelist():
            root = ET.fromstring(archive.read("xl/sharedStrings.xml"))
            for si in root.findall("m:si", ns):
                shared.append("".join(si.itertext()).strip())
        sheets = sorted(
            name for name in archive.namelist()
            if name.startswith("xl/worksheets/sheet") and name.endswith(".xml")
        )
        for sheet_no, name in enumerate(sheets, start=1):
            root = ET.fromstring(archive.read(name))
            values.append(f"[시트 {sheet_no}]")
            for row in root.findall(".//m:row", ns):
                row_values = []
                for cell in row.findall("m:c", ns):
                    cell_type = cell.get("t", "")
                    value_node = cell.find("m:v", ns)
                    inline = cell.find("m:is", ns)
                    value = ""
                    if cell_type == "s" and value_node is not None:
                        try:
                            value = shared[int(value_node.text or "0")]
                        except Exception:
                            value = value_node.text or ""
                    elif cell_type == "inlineStr" and inline is not None:
                        value = "".join(inline.itertext())
                    elif value_node is not None:
                        value = value_node.text or ""
                    row_values.append(norm(value))
                if any(row_values):
                    values.append(" | ".join(row_values))
    return _clean_extracted_text("\n".join(values), config.attachment_max_text_chars)


def extract_hwp_text(path: Path, config: PipelineConfig) -> str:
    import olefile
    ole = olefile.OleFileIO(str(path))
    try:
        header = ole.openstream("FileHeader").read()
        compressed = len(header) > 36 and bool(header[36] & 1)
        sections = []
        for entry in ole.listdir():
            joined = "/".join(entry)
            if joined.startswith("BodyText/Section"):
                try:
                    number = int(re.search(r"Section(\d+)", joined).group(1))
                except Exception:
                    number = 0
                sections.append((number, joined))
        texts: list[str] = []
        for _, stream_name in sorted(sections):
            data = ole.openstream(stream_name).read()
            if compressed:
                try:
                    data = zlib.decompress(data, -15)
                except Exception:
                    continue
            offset = 0
            while offset + 4 <= len(data):
                header_value = struct.unpack_from("<I", data, offset)[0]
                offset += 4
                tag_id = header_value & 0x3FF
                size = (header_value >> 20) & 0xFFF
                if size == 0xFFF:
                    if offset + 4 > len(data):
                        break
                    size = struct.unpack_from("<I", data, offset)[0]
                    offset += 4
                payload = data[offset: offset + size]
                offset += size
                if tag_id == 67:
                    text = payload.decode("utf-16le", errors="ignore")
                    text = re.sub(r"[\x00-\x1f]", " ", text)
                    if norm(text):
                        texts.append(norm(text))
        return _clean_extracted_text("\n".join(texts), config.attachment_max_text_chars)
    finally:
        ole.close()


def extract_attachment_text(path: Path, detected_format: str, config: PipelineConfig) -> tuple[str, str, str]:
    if not config.extract_attachment_text:
        return "", "disabled", ""
    try:
        fmt = detected_format.upper()
        if fmt == "PDF":
            text = extract_pdf_text(path, config)
        elif fmt in {"DOCX", "PPTX"}:
            text = extract_docx_or_pptx_text(path, config)
        elif fmt == "HWPX":
            text = extract_hwpx_text(path, config)
        elif fmt == "XLSX":
            text = extract_xlsx_text(path, config)
        elif fmt == "HWP":
            text = extract_hwp_text(path, config)
        elif fmt in {"CSV", "TXT"}:
            raw = path.read_bytes()
            for encoding in ("utf-8-sig", "cp949", "euc-kr", "utf-8"):
                try:
                    text = raw.decode(encoding)
                    break
                except Exception:
                    text = ""
            text = _clean_extracted_text(text, config.attachment_max_text_chars)
        else:
            return "", "unsupported", f"지원하지 않는 텍스트 추출 형식: {fmt}"
        if not text:
            return "", "empty_or_scanned", "텍스트가 없거나 스캔 문서일 수 있음"
        return text, "success", ""
    except Exception as error:
        return "", "failed", f"{type(error).__name__}: {error}"


def classify_attachment_policy(attachment: dict[str, Any], config: PipelineConfig) -> dict[str, Any]:
    name = " ".join([
        attachment.get("display_name", ""),
        attachment.get("row_context", ""),
        attachment.get("filename", ""),
    ])
    text = attachment.get("extracted_text", "")
    status = attachment.get("text_extraction_status", "")
    downloaded = attachment.get("download_status") == "downloaded"
    detected = attachment.get("detected_format", attachment.get("file_format", "")).upper()

    if not downloaded:
        return {
            "processing_policy": "metadata_only",
            "indexable": False,
            "classification_reason": "원본 파일 미다운로드",
            "needs_review": attachment.get("download_status") == "failed",
        }
    if any(keyword in name for keyword in FORM_NO_INDEX_KEYWORDS):
        return {
            "processing_policy": "download_no_index",
            "indexable": False,
            "classification_reason": "신청서·동의서·위임장 등 작성용 서식",
            "needs_review": False,
        }
    if status != "success":
        return {
            "processing_policy": "download_no_index" if detected in SUPPORTED_ATTACHMENT_TEXT_FORMATS else "metadata_only",
            "indexable": False,
            "classification_reason": "텍스트 추출 실패·미지원 또는 스캔 문서",
            "needs_review": True,
        }
    if any(keyword in name for keyword in ATTACHMENT_INDEX_KEYWORDS):
        return {
            "processing_policy": "download_and_index",
            "indexable": True,
            "classification_reason": "안내·절차·목록형 자료",
            "needs_review": False,
        }
    if len(text) >= max(config.attachment_index_min_chars, 400):
        return {
            "processing_policy": "download_and_index",
            "indexable": True,
            "classification_reason": "충분한 설명 텍스트가 있는 공개 자료",
            "needs_review": True,
        }
    return {
        "processing_policy": "download_no_index",
        "indexable": False,
        "classification_reason": "짧은 서식 또는 설명 가치가 낮은 파일",
        "needs_review": len(text) >= config.attachment_index_min_chars,
    }


def _write_stream_to_file(response: requests.Response, output: Path, config: PipelineConfig) -> int:
    ensure_parent(output)
    written = 0
    with output.open("wb") as file:
        for chunk in response.iter_content(64 * 1024):
            if not chunk:
                continue
            written += len(chunk)
            if written > config.max_asset_bytes:
                raise ValueError("첨부파일 제한 용량 초과")
            file.write(chunk)
    return written


def download_direct_assets(
    session: requests.Session,
    document: dict[str, Any],
    row: dict[str, str],
    paths: OutputPaths,
    config: PipelineConfig,
) -> dict[str, list[dict[str, Any]]]:
    result = {"attachments": [], "images": [], "failures": []}
    base_dir = paths.raw_assets / safe_name(row["business_function"]) / row["url_id"]

    attachment_download_enabled = (
        document.get("policy", {}).get("attachment_download_policy") == "O"
    )
    if config.download_direct_attachments and attachment_download_enabled:
        for attachment in document.get("attachments", []):
            if attachment.get("download_method") != "href" or not attachment.get("url"):
                continue
            try:
                response = session.get(
                    attachment["url"], timeout=config.request_timeout_seconds,
                    stream=True, allow_redirects=True,
                )
                response.raise_for_status()
                content_type = response.headers.get("Content-Type", "")
                filename = (
                    _filename_from_content_disposition(response.headers.get("Content-Disposition", ""))
                    or Path(urlparse(response.url).path).name
                    or attachment["display_name"]
                )
                filename = f"{attachment['attachment_id']}_{safe_name(filename)}"
                output = base_dir / "attachments" / filename
                _write_stream_to_file(response, output, config)
                inspection = inspect_downloaded_attachment(output, attachment.get("file_format", ""))
                if not inspection["signature_valid"]:
                    raise ValueError("파일 대신 HTML 오류 페이지 또는 빈 응답을 받음")
                attachment.update({
                    "download_status": "downloaded",
                    "filename": filename,
                    "local_path": str(output.relative_to(paths.root)),
                    "mime_type": content_type.split(";", 1)[0].strip(),
                    "official_download_url": response.url,
                    "download_url_final": response.url,
                    "user_action_url": response.url,
                    "delivery_mode": "direct_official_download",
                    **inspection,
                })
                result["attachments"].append(attachment)
            except Exception as error:
                attachment.update({
                    "download_status": "failed",
                    "processing_policy": "metadata_only",
                    "indexable": False,
                    "needs_review": True,
                    "error": f"{type(error).__name__}: {error}",
                })
                result["failures"].append({
                    "type": "attachment", "name": attachment.get("display_name", ""),
                    "error": attachment["error"],
                })

    if config.download_images:
        for image in document.get("images", []):
            if image.get("image_role") == "decorative":
                continue
            try:
                response = session.get(image["url"], timeout=config.request_timeout_seconds, stream=True)
                response.raise_for_status()
                filename = image.get("filename") or Path(urlparse(response.url).path).name
                output = base_dir / "images" / safe_name(filename)
                _write_stream_to_file(response, output, config)
                result["images"].append({
                    **image, "local_path": str(output.relative_to(paths.root)),
                    "size_bytes": output.stat().st_size, "sha256": sha256_bytes(output.read_bytes()),
                })
            except Exception as error:
                result["failures"].append({
                    "type": "image", "url": image.get("url", ""),
                    "error": f"{type(error).__name__}: {error}",
                })
    return result


def download_js_attachments(
    documents: list[dict[str, Any]], paths: OutputPaths, config: PipelineConfig
) -> list[dict[str, Any]]:
    records = []
    for document in documents:
        if document.get("policy", {}).get("attachment_download_policy") != "O":
            continue
        for attachment in document.get("attachments", []):
            if attachment.get("download_method") == "gfn_downloadFile":
                records.append((document, attachment))
    if not records or not config.download_js_attachments_with_playwright:
        return []
    ensure_playwright()
    from playwright.sync_api import sync_playwright
    results: list[dict[str, Any]] = []
    with sync_playwright() as playwright:
        browser = playwright.chromium.launch(headless=True)
        context = browser.new_context(accept_downloads=True, user_agent=config.user_agent, locale="ko-KR")
        grouped: dict[str, list[tuple[dict[str, Any], dict[str, Any]]]] = {}
        for document, attachment in records:
            grouped.setdefault(document["source_url"], []).append((document, attachment))
        for source_url, items in grouped.items():
            page = context.new_page()
            try:
                page.goto(source_url, wait_until="domcontentloaded", timeout=config.request_timeout_seconds * 1000)
                page.wait_for_function("typeof gfn_downloadFile === 'function'", timeout=config.request_timeout_seconds * 1000)
                for document, attachment in items:
                    try:
                        with page.expect_download(timeout=60_000) as info:
                            page.evaluate(
                                "([a,b]) => gfn_downloadFile(a,b)",
                                [attachment["token1"], attachment["token2"]],
                            )
                        download = info.value
                        filename = f"{attachment['attachment_id']}_{safe_name(download.suggested_filename)}"
                        output = (
                            paths.raw_assets / safe_name(document["business_function"]) /
                            document["doc_id"] / "attachments" / filename
                        )
                        ensure_parent(output)
                        download.save_as(str(output))
                        inspection = inspect_downloaded_attachment(output, attachment.get("file_format", ""))
                        if not inspection["signature_valid"]:
                            raise ValueError("파일 대신 HTML 오류 페이지 또는 빈 응답을 받음")
                        attachment.update({
                            "download_status": "downloaded",
                            "filename": filename,
                            "local_path": str(output.relative_to(paths.root)),
                            "source_page_url": source_url,
                            "official_download_url": None,
                            "user_action_url": source_url,
                            "delivery_mode": "official_download_page",
                            **inspection,
                        })
                        results.append(attachment)
                    except Exception as error:
                        attachment.update({
                            "download_status": "failed",
                            "processing_policy": "metadata_only",
                            "indexable": False,
                            "needs_review": True,
                            "error": f"{type(error).__name__}: {error}",
                        })
            finally:
                page.close()
        context.close()
        browser.close()
    return results


def ensure_source_page_action(document: dict[str, Any], row: dict[str, str]) -> None:
    if document.get("actions") or not row.get("action_policy", "").startswith("O"):
        return
    expected = [value for value in row.get("expected_action_types", "").split("|") if value]
    action_type = expected[0] if expected else "related_service"
    label_suffix = {
        "self_check": "자가진단", "lookup": "조회", "apply": "신청",
        "progress": "진행상황 조회", "consult": "상담",
    }.get(action_type, "바로가기")
    document["actions"] = [{
        "action_id": sha256_text(f"{row['url_id']}|source-page|{action_type}")[:16],
        "label": f"{row['page_title']} {label_suffix}",
        "url": row["source_url"],
        "action_type": action_type,
        "source_tag": "source_page",
        "auth_required": row.get("action_auth", row.get("auth_required", "")),
        "official_domain": allowed_action_domain(row["source_url"]),
        "requires_review": False,
    }]


def sync_attachment_actions(document: dict[str, Any]) -> None:
    actions = [action for action in document.get("actions", []) if action.get("source_tag") != "attachment"]
    if document.get("policy", {}).get("attachment_user_delivery_policy") == "none":
        document["actions"] = actions
        return
    seen = {(action.get("url"), action.get("label")) for action in actions}
    for attachment in document.get("attachments", []):
        direct = bool(attachment.get("official_download_url"))
        url = attachment.get("official_download_url") or attachment.get("source_page_url") or document["source_url"]
        # 안정적인 공식 파일 URL이 있으면 직접 다운로드, JS 토큰 방식이면 공식 게시 페이지를 제공합니다.
        suffix = "공식 파일 다운로드" if direct else "공식 다운로드 페이지"
        label = f"{attachment.get('display_name', '첨부파일')} {suffix}"
        key = (url, label)
        if key in seen:
            continue
        seen.add(key)
        actions.append({
            "action_id": attachment["attachment_id"],
            "label": label,
            "url": url,
            "action_type": "download",
            "source_tag": "attachment",
            "attachment_id": attachment["attachment_id"],
            "download_method": attachment.get("download_method"),
            "delivery_mode": "direct_official_download" if direct else "official_download_page",
            "auth_required": "불필요",
            "official_domain": allowed_action_domain(url),
            "requires_review": attachment.get("needs_review", False),
        })
    document["actions"] = actions


def attachment_to_document(
    parent: dict[str, Any], attachment: dict[str, Any], paths: OutputPaths, config: PipelineConfig
) -> dict[str, Any]:
    local_path = paths.root / attachment["local_path"]
    detected = attachment.get("detected_format") or detect_attachment_format(local_path, attachment.get("file_format", ""))
    text, extraction_status, extraction_error = extract_attachment_text(local_path, detected, config)
    attachment.update({
        "detected_format": detected,
        "extracted_text": text,
        "extracted_text_chars": len(text),
        "text_extraction_status": extraction_status,
        "text_extraction_error": extraction_error,
    })
    attachment.update(classify_attachment_policy(attachment, config))

    title = attachment.get("display_name") or attachment.get("filename") or "첨부파일"
    markdown = f"# {title}\n\n{text}".strip() if text else f"# {title}"
    doc_id = f"{parent['doc_id']}-ATT-{attachment['attachment_id'][:8]}"
    document = {
        "doc_id": doc_id,
        "parent_doc_id": parent["doc_id"],
        "attachment_id": attachment["attachment_id"],
        "record_type": "attachment",
        "indexable": attachment["indexable"],
        "rag_index_mode": "attachment" if attachment["indexable"] else "none",
        "title": title,
        "source_url": attachment.get("source_page_url") or parent["source_url"],
        "official_download_url": attachment.get("official_download_url"),
        "user_action_url": attachment.get("user_action_url") or attachment.get("source_page_url"),
        "site_name": parent.get("site_name", ""),
        "business_function": parent["business_function"],
        "target_business_function": parent.get("target_business_function", parent["business_function"]),
        "page_type": "attachment",
        "file_format": detected,
        "mime_type": attachment.get("mime_type", ""),
        "filename": attachment.get("filename", ""),
        "local_path": attachment.get("local_path", ""),
        "size_bytes": attachment.get("size_bytes", 0),
        "sha256": attachment.get("sha256", ""),
        "processing_policy": attachment["processing_policy"],
        "classification_reason": attachment["classification_reason"],
        "needs_review": attachment["needs_review"],
        "text_extraction_status": extraction_status,
        "content_text": text,
        "content_markdown": markdown,
        "blocks": [{"type": "paragraph", "text": text}] if text else [],
        "actions": [],
        "attachments": [],
        "images": [],
        "videos": [],
        "supplementary_links": [],
        "links": [],
        "quality": {
            "status": "review" if attachment["needs_review"] else "pass",
            "reasons": [attachment["classification_reason"]] if attachment["needs_review"] else [],
            "parsed_text_chars": len(text),
            "faq_count": 0,
            "table_count": 0,
        },
        "parsed_at": now_kst_iso(),
    }
    folder = paths.parsed_attachments / safe_name(parent["business_function"]) / parent["doc_id"]
    md_path = folder / f"{doc_id}.md"
    json_path = folder / f"{doc_id}.json"
    ensure_parent(md_path)
    md_path.write_text(markdown, encoding="utf-8")
    write_json(json_path, document)
    attachment["parsed_markdown_path"] = str(md_path.relative_to(paths.root))
    attachment["parsed_json_path"] = str(json_path.relative_to(paths.root))
    return document


def process_attachment_documents(
    page_documents: list[dict[str, Any]], paths: OutputPaths, config: PipelineConfig
) -> tuple[list[dict[str, Any]], pd.DataFrame]:
    attachment_documents: list[dict[str, Any]] = []
    manifest_rows: list[dict[str, Any]] = []
    for parent in page_documents:
        parent_rag_policy = parent.get("policy", {}).get("attachment_rag_policy", "none")
        for attachment in parent.get("attachments", []):
            if (
                attachment.get("download_status") == "downloaded"
                and config.create_attachment_documents
                and parent_rag_policy == "auto_classify_guide_only"
            ):
                try:
                    attachment_document = attachment_to_document(parent, attachment, paths, config)
                    attachment_documents.append(attachment_document)
                except Exception as error:
                    attachment.update({
                        "processing_policy": "metadata_only", "indexable": False,
                        "needs_review": True,
                        "text_extraction_status": "failed",
                        "text_extraction_error": f"{type(error).__name__}: {error}",
                    })
            else:
                attachment.update(classify_attachment_policy(attachment, config))

            if parent_rag_policy != "auto_classify_guide_only":
                attachment.update({
                    "processing_policy": "download_no_index" if attachment.get("download_status") == "downloaded" else "metadata_only",
                    "indexable": False,
                    "classification_reason": "URL 정책표에서 첨부파일 RAG 인덱싱을 사용하지 않음",
                })
            manifest_rows.append({
                "parent_doc_id": parent["doc_id"],
                "business_function": parent["business_function"],
                "attachment_download_policy": parent.get("policy", {}).get("attachment_download_policy", "X"),
                "attachment_rag_policy": parent.get("policy", {}).get("attachment_rag_policy", "none"),
                "attachment_user_delivery_policy": parent.get("policy", {}).get("attachment_user_delivery_policy", "none"),
                "attachment_id": attachment.get("attachment_id", ""),
                "display_name": attachment.get("display_name", ""),
                "file_format": attachment.get("file_format", ""),
                "detected_format": attachment.get("detected_format", ""),
                "download_method": attachment.get("download_method", ""),
                "download_status": attachment.get("download_status", ""),
                "processing_policy": attachment.get("processing_policy", ""),
                "indexable": attachment.get("indexable", False),
                "needs_review": attachment.get("needs_review", False),
                "classification_reason": attachment.get("classification_reason", ""),
                "text_extraction_status": attachment.get("text_extraction_status", ""),
                "extracted_text_chars": attachment.get("extracted_text_chars", 0),
                "source_page_url": attachment.get("source_page_url", parent["source_url"]),
                "official_download_url": attachment.get("official_download_url"),
                "user_action_url": attachment.get("user_action_url", parent["source_url"]),
                "local_path": attachment.get("local_path", ""),
                "size_bytes": attachment.get("size_bytes", 0),
                "sha256": attachment.get("sha256", ""),
                "error": attachment.get("error") or attachment.get("text_extraction_error", ""),
            })
        sync_attachment_actions(parent)
    manifest_df = pd.DataFrame(manifest_rows)
    manifest_df.to_csv(paths.processed / "attachment_manifest.csv", index=False, encoding="utf-8-sig")
    with (paths.processed / "attachment_manifest.jsonl").open("w", encoding="utf-8") as file:
        for row in manifest_rows:
            file.write(json.dumps(row, ensure_ascii=False) + "\n")
    if not manifest_df.empty:
        manifest_df[manifest_df["needs_review"].fillna(False)].to_csv(
            paths.quality / "attachment_review.csv", index=False, encoding="utf-8-sig"
        )
    return attachment_documents, manifest_df


def _chunk_plain_len(text: str) -> int:
    return len(re.sub(r"[#|\-\s]", "", text))


def _noise_chunk(text: str) -> bool:
    return any(fragment in text for fragment in NOISE_CONTAINS_TEXTS)


def split_text_lossless(text: str, max_chars: int) -> list[str]:
    text = text.strip()
    if len(text) <= max_chars:
        return [text] if text else []
    paragraphs = [value.strip() for value in re.split(r"\n\s*\n", text) if value.strip()]
    chunks: list[str] = []
    current = ""
    for paragraph in paragraphs:
        if len(paragraph) > max_chars:
            sentences = [value.strip() for value in re.split(r"(?<=[.!?다요])\s+", paragraph) if value.strip()]
        else:
            sentences = [paragraph]
        for sentence in sentences:
            candidate = f"{current}\n\n{sentence}".strip() if current else sentence
            if current and len(candidate) > max_chars:
                chunks.append(current)
                current = sentence
            elif len(sentence) > max_chars and not current:
                for start in range(0, len(sentence), max_chars):
                    chunks.append(sentence[start:start + max_chars])
                current = ""
            else:
                current = candidate
    if current:
        chunks.append(current)
    return chunks


def structure_aware_chunks(document: dict[str, Any], config: PipelineConfig) -> list[dict[str, Any]]:
    """짧은 FAQ와 짧은 본문을 삭제하지 않는 무손실 청킹입니다."""
    if not document.get("indexable", True):
        return []

    intermediate: list[dict[str, str]] = []
    current_heading = ""
    current_parts: list[str] = []

    def add_section(content: str, title: str) -> None:
        content = content.strip()
        if not content or _noise_chunk(content):
            return
        for part in split_text_lossless(content, config.chunk_max_chars):
            intermediate.append({"section_title": title, "chunk_type": "section", "content": part})

    def current_is_heading_only() -> bool:
        content = "\n\n".join(part for part in current_parts if part).strip()
        return bool(current_heading and content == f"### {current_heading}")

    def flush() -> None:
        nonlocal current_parts
        content = "\n\n".join(part for part in current_parts if part).strip()
        current_parts = []
        add_section(content, current_heading)

    for block in document.get("blocks", []):
        kind = block.get("type")
        if kind == "heading":
            flush()
            current_heading = block.get("text", "")
            current_parts = [render_blocks([block])]
            continue
        if kind == "faq":
            if current_is_heading_only():
                current_parts = []
            else:
                flush()
            content = render_blocks([block]).strip()
            if content and not _noise_chunk(content):
                intermediate.append({
                    "section_title": block.get("question", ""),
                    "chunk_type": "faq", "content": content,
                })
            continue
        if kind == "table":
            pending_heading = current_heading
            # 표 앞의 설명·목록은 먼저 보존하되, 제목만 있는 경우에는
            # 제목을 별도 청크로 만들지 않고 표 청크에 붙입니다.
            if current_is_heading_only():
                current_parts = []
            else:
                flush()
            for content in split_table(block, config.chunk_max_chars, pending_heading):
                if content.strip() and not _noise_chunk(content):
                    intermediate.append({
                        "section_title": pending_heading,
                        "chunk_type": "table", "content": content.strip(),
                    })
            continue
        block_text = render_blocks([block])
        candidate = "\n\n".join(current_parts + [block_text]).strip()
        if current_parts and len(candidate) > config.chunk_max_chars:
            flush()
            if current_heading:
                current_parts = [f"### {current_heading}"]
        current_parts.append(block_text)
    flush()

    # 짧은 일반 섹션은 삭제하지 않고 앞 섹션과 결합하거나 그대로 유지합니다.
    merged: list[dict[str, str]] = []
    for item in intermediate:
        if (
            item["chunk_type"] == "section"
            and _chunk_plain_len(item["content"]) < config.min_chunk_chars
            and merged
            and merged[-1]["chunk_type"] == "section"
            and len(merged[-1]["content"] + "\n\n" + item["content"]) <= config.chunk_max_chars
        ):
            merged[-1]["content"] += "\n\n" + item["content"]
        else:
            merged.append(item)

    # 문서 맨 앞의 짧은 제목 청크는 다음 일반 섹션과 결합합니다.
    if (
        len(merged) >= 2
        and merged[0]["chunk_type"] == "section"
        and _chunk_plain_len(merged[0]["content"]) < config.min_chunk_chars
        and merged[1]["chunk_type"] == "section"
        and len(merged[0]["content"] + "\n\n" + merged[1]["content"]) <= config.chunk_max_chars
    ):
        merged[1]["content"] = merged[0]["content"] + "\n\n" + merged[1]["content"]
        merged.pop(0)

    # 인덱싱 문서가 내용이 있는데 청크 0개가 되는 것을 방지합니다.
    if not merged and document.get("content_markdown", "").strip():
        for part in split_text_lossless(document["content_markdown"], config.chunk_max_chars):
            merged.append({"section_title": "", "chunk_type": "section", "content": part})

    records: list[dict[str, Any]] = []
    seen_hashes: set[str] = set()
    action_types = sorted({
        action.get("action_type", "") for action in document.get("actions", [])
        if action.get("action_type")
    })
    for item in merged:
        content = item["content"].strip()
        content_hash = sha256_text(content)
        if content_hash in seen_hashes:
            continue
        seen_hashes.add(content_hash)
        records.append({
            "chunk_id": f"{document['doc_id']}_chunk_{len(records):03d}",
            "parent_doc_id": document.get("parent_doc_id", document["doc_id"]),
            "document_id": document["doc_id"],
            "attachment_id": document.get("attachment_id"),
            "record_type": document.get("record_type", "page"),
            "chunk_index": len(records),
            "title": document["title"],
            "section_title": item["section_title"],
            "chunk_type": item["chunk_type"],
            "business_function": document["business_function"],
            "target_business_function": document["target_business_function"],
            "page_type": document["page_type"],
            "rag_index_mode": document.get("rag_index_mode", "full"),
            "source_url": document["source_url"],
            "official_download_url": document.get("official_download_url"),
            "available_action_types": action_types,
            "content": content,
            "content_hash": content_hash,
        })
    return records


def _normalized_coverage_text(value: str) -> str:
    return re.sub(r"\s+", "", re.sub(r"[#|`*\-]", "", value or ""))


def atomic_index_units(document: dict[str, Any]) -> list[str]:
    units: list[str] = []
    for block in iter_blocks_recursive(document.get("blocks", [])):
        kind = block.get("type")
        if kind == "heading":
            continue
        if kind == "faq":
            value = render_blocks([block])
            if value:
                units.append(value)
        elif kind == "table":
            headers = block.get("headers", [])
            for row in block.get("rows", []):
                units.append(" ".join(headers + row))
        else:
            value = blocks_text([block])
            if value:
                units.append(value)
    if not units and document.get("content_text"):
        units.append(document["content_text"])
    return units


def calculate_chunk_coverage(document: dict[str, Any], chunks: list[dict[str, Any]]) -> float:
    """문서의 고유 단어·숫자가 청크에 보존됐는지 계산합니다."""
    token_pattern = re.compile(r"[가-힣A-Za-z]{2,}|[0-9]+")
    source_tokens = set(token_pattern.findall(document.get("content_text", "").lower()))
    if not source_tokens:
        return 1.0
    chunk_text = "\n".join(chunk.get("content", "") for chunk in chunks).lower()
    chunk_tokens = set(token_pattern.findall(chunk_text))
    return round(len(source_tokens & chunk_tokens) / len(source_tokens), 4)


def run_pipeline(
    review_csv_path: str | Path,
    runtime_root: str | Path,
    config: PipelineConfig | None = None,
) -> dict[str, Any]:
    config = config or PipelineConfig()
    validate_media_policy(config)
    runtime_root = Path(runtime_root)
    paths = create_output_paths(runtime_root)
    session = create_session(config)

    review_df = pd.read_csv(review_csv_path, encoding="utf-8-sig", dtype=str).fillna("")
    manifest_df = normalize_review_manifest(review_df)
    manifest_df.to_csv(paths.manifest / "manifest_full_42.csv", index=False, encoding="utf-8-sig")
    review_df.to_csv(paths.manifest / "review_policy_42.csv", index=False, encoding="utf-8-sig")

    target_df = manifest_df.copy()
    if config.select_business_functions:
        target_df = target_df[target_df["business_function"].isin(config.select_business_functions)]
    if config.run_only_url_ids:
        unknown = sorted(set(config.run_only_url_ids) - set(manifest_df["url_id"]))
        if unknown:
            raise ValueError(f"Manifest에 없는 URL ID: {unknown}")
        target_df = target_df[target_df["url_id"].isin(config.run_only_url_ids)]
    if config.max_urls is not None:
        target_df = target_df.head(config.max_urls)
    target_df.to_csv(paths.manifest / "manifest_run.csv", index=False, encoding="utf-8-sig")

    page_documents: list[dict[str, Any]] = []
    run_results: list[dict[str, Any]] = []
    crawl_jsonl = paths.logs / "crawl_results.jsonl"

    for _, series in tqdm(target_df.iterrows(), total=len(target_df), desc="KDIC 42개 정책 기반 수집"):
        row = row_to_dict(series)
        started = now_kst_iso()
        try:
            if row["normalized_decision"] == "link_only":
                document = build_link_only_document(row)
                save_document(paths, document, row)
                page_documents.append(document)
                result = {
                    "url_id": row["url_id"], "business_function": row["business_function"],
                    "page_title": row["page_title"], "status": "link_metadata_created",
                    "status_code": None, "content_chars": 0, "faq_count": 0,
                    "table_count": 0, "action_count": 1, "attachment_count": 0,
                    "downloaded_attachment_count": 0,
                    "pagination_detected": False, "pagination_is_complete": True,
                    "pagination_page_count": 0, "error_type": "", "error": "",
                }
            else:
                first = fetch_url(session, row["source_url"], config)
                html_path, meta_path = save_fetch_result(paths, row, first)
                if not first.ok:
                    raise requests.HTTPError(f"HTTP {first.status_code}: {first.final_url}")
                if "html" not in first.content_type.lower():
                    raise ValueError(f"HTML 응답이 아님: {first.content_type}")
                first_document = parse_html_document(first.body, first.final_url, row, first.encoding)
                document, pagination = collect_paginated_document(
                    session, first, first_document, row, paths, config
                )
                ensure_source_page_action(document, row)
                document["quality"] = build_quality(document, document.get("content_text", ""), row)
                assets = download_direct_assets(session, document, row, paths, config)
                document["downloaded_assets"] = assets
                md_path, json_path = save_document(paths, document, row)
                page_documents.append(document)
                result = {
                    "url_id": row["url_id"], "business_function": row["business_function"],
                    "page_title": document["title"], "status": (
                        "paginated_full_collection_created" if pagination.get("pagination_detected") and pagination.get("is_complete")
                        else "pagination_collection_incomplete" if pagination.get("pagination_detected")
                        else "parse_success"
                    ),
                    "status_code": first.status_code,
                    "content_chars": len(document.get("content_text", "")),
                    "quality_status": document.get("quality", {}).get("status", ""),
                    "quality_reasons": " | ".join(document.get("quality", {}).get("reasons", [])),
                    "retention_ratio": document.get("quality", {}).get("retention_ratio", 0),
                    "faq_count": document.get("quality", {}).get("faq_count", 0),
                    "table_count": document.get("quality", {}).get("table_count", 0),
                    "action_count": len(document.get("actions", [])),
                    "attachment_count": len(document.get("attachments", [])),
                    "downloaded_attachment_count": len(assets.get("attachments", [])),
                    "image_count": len(document.get("images", [])),
                    "video_count": len(document.get("videos", [])),
                    "supplementary_link_count": len(document.get("supplementary_links", [])),
                    "asset_failure_count": len(assets.get("failures", [])),
                    "pagination_detected": pagination.get("pagination_detected", False),
                    "pagination_is_complete": pagination.get("is_complete", True),
                    "pagination_page_count": pagination.get("fetched_page_count", 1),
                    "pagination_failure_count": len(pagination.get("failures", [])),
                    "raw_html_path": str(html_path.relative_to(paths.root)),
                    "response_meta_path": str(meta_path.relative_to(paths.root)),
                    "parsed_markdown_path": str(md_path.relative_to(paths.root)),
                    "parsed_json_path": str(json_path.relative_to(paths.root)),
                    "error_type": "", "error": "",
                }
        except Exception as error:
            result = {
                "url_id": row["url_id"], "business_function": row["business_function"],
                "page_title": row["page_title"], "status": "failed", "status_code": None,
                "content_chars": 0, "faq_count": 0, "table_count": 0, "action_count": 0,
                "attachment_count": 0, "downloaded_attachment_count": 0,
                "pagination_detected": False, "pagination_is_complete": False,
                "pagination_page_count": 0,
                "error_type": type(error).__name__, "error": str(error),
            }
        result["started_at"] = started
        result["finished_at"] = now_kst_iso()
        run_results.append(result)
        append_jsonl(crawl_jsonl, result)
        if row["normalized_decision"] != "link_only":
            time.sleep(config.request_delay_seconds)

    # JavaScript 기반 공개 첨부파일도 다운로드합니다.
    download_js_attachments(page_documents, paths, config)

    # 모든 첨부파일을 검증·텍스트 추출·정책 분류합니다.
    attachment_documents, attachment_manifest_df = process_attachment_documents(
        page_documents, paths, config
    )
    documents = page_documents + attachment_documents

    # 영상·웹툰은 다운로드하지 않고 공식 게시 페이지 참고 링크 목록으로 저장합니다.
    supplementary_rows: list[dict[str, Any]] = []
    for parent in page_documents:
        for link in parent.get("supplementary_links", []):
            supplementary_rows.append({
                "parent_doc_id": parent["doc_id"],
                "business_function": parent["business_function"],
                "title": parent["title"],
                **link,
            })
    supplementary_df = pd.DataFrame(supplementary_rows)
    supplementary_df.to_csv(
        paths.processed / "supplementary_links.csv",
        index=False,
        encoding="utf-8-sig",
    )
    with (paths.processed / "supplementary_links.jsonl").open("w", encoding="utf-8") as file:
        for row in supplementary_rows:
            file.write(json.dumps(row, ensure_ascii=False) + "\n")

    # 첨부파일 상태와 Action이 갱신되었으므로 페이지 JSON을 다시 저장합니다.
    by_id = {row["url_id"]: row_to_dict(row) for _, row in target_df.iterrows()}
    for document in page_documents:
        save_document(paths, document, by_id[document["doc_id"]])

    results_df = pd.DataFrame(run_results)
    if not results_df.empty:
        downloaded_by_parent = {}
        for parent in page_documents:
            downloaded_by_parent[parent["doc_id"]] = sum(
                1 for item in parent.get("attachments", [])
                if item.get("download_status") == "downloaded"
            )
        results_df["downloaded_attachment_count"] = results_df["url_id"].map(downloaded_by_parent).fillna(0).astype(int)
    results_df.to_csv(paths.logs / "crawl_results.csv", index=False, encoding="utf-8-sig")

    documents_path = paths.processed / "documents.jsonl"
    with documents_path.open("w", encoding="utf-8") as file:
        for document in documents:
            file.write(json.dumps(document, ensure_ascii=False) + "\n")

    chunks: list[dict[str, Any]] = []
    chunks_by_doc: dict[str, list[dict[str, Any]]] = {}
    if config.create_chunks:
        for document in documents:
            doc_chunks = structure_aware_chunks(document, config)
            chunks.extend(doc_chunks)
            chunks_by_doc[document["doc_id"]] = doc_chunks
    chunks_path = paths.processed / "chunks.jsonl"
    with chunks_path.open("w", encoding="utf-8") as file:
        for chunk in chunks:
            file.write(json.dumps(chunk, ensure_ascii=False) + "\n")

    quality_rows: list[dict[str, Any]] = []
    validations: list[dict[str, Any]] = []
    for document in documents:
        quality = document.get("quality", {})
        pagination = document.get("pagination_collection", {}) or {}
        doc_chunks = chunks_by_doc.get(document["doc_id"], [])
        faq_count = count_block_type(document.get("blocks", []), "faq")
        faq_chunk_count = sum(1 for chunk in doc_chunks if chunk["chunk_type"] == "faq")
        coverage = calculate_chunk_coverage(document, doc_chunks) if document.get("indexable", True) else 1.0
        chunk_complete = (
            (not document.get("indexable", True))
            or (
                len(doc_chunks) >= 1
                and faq_count == faq_chunk_count
                and coverage >= 0.98
            )
        )
        reasons = list(quality.get("reasons", []))
        if not chunk_complete:
            reasons.append("청킹 데이터 손실 가능성")
        quality_status = "fail" if not chunk_complete else quality.get("status", "pass")
        quality_rows.append({
            "url_id": document["doc_id"],
            "parent_doc_id": document.get("parent_doc_id", document["doc_id"]),
            "record_type": document.get("record_type", "page"),
            "business_function": document["business_function"],
            "title": document["title"],
            "indexable": document.get("indexable", True),
            "rag_index_mode": document.get("rag_index_mode", ""),
            "quality_status": quality_status,
            "quality_reasons": " | ".join(dict.fromkeys(reasons)),
            "content_chars": len(document.get("content_text", "")),
            "chunk_count": len(doc_chunks),
            "chunk_coverage_ratio": coverage,
            "faq_count": faq_count,
            "faq_chunk_count": faq_chunk_count,
            "table_count": count_block_type(document.get("blocks", []), "table"),
            "action_count": len(document.get("actions", [])),
            "attachment_count": len(document.get("attachments", [])),
            "video_count": len(document.get("videos", [])),
            "supplementary_link_count": len(document.get("supplementary_links", [])),
            "pagination_detected": pagination.get("pagination_detected", False),
            "pagination_is_complete": pagination.get("is_complete", True),
            "pagination_page_count": pagination.get("fetched_page_count", 1),
            "processing_policy": document.get("processing_policy", ""),
            "needs_review": document.get("needs_review", False),
        })
        if document.get("indexable", True):
            validations.extend([
                {
                    "validation": f"{document['doc_id']} 인덱싱 문서 청크 존재",
                    "passed": len(doc_chunks) >= 1,
                },
                {
                    "validation": f"{document['doc_id']} FAQ 청크 무손실",
                    "passed": faq_count == faq_chunk_count,
                },
                {
                    "validation": f"{document['doc_id']} 청크 본문 보존율",
                    "passed": coverage >= 0.98,
                    "value": coverage,
                },
            ])

    quality_df = pd.DataFrame(quality_rows)
    quality_df.to_csv(paths.quality / "quality_report.csv", index=False, encoding="utf-8-sig")

    by_id = {document["doc_id"]: document for document in page_documents}
    if "DP-001" in by_id:
        validations.append({"validation": "DP-001 1억원", "passed": "1억원" in by_id["DP-001"].get("content_text", "")})
    if "UN-003" in by_id:
        text = by_id["UN-003"].get("content_text", "")
        validations.append({"validation": "UN-003 미수령금 종류", "passed": all(k in text for k in ["예금보험금", "개산지급금", "파산배당금"])})
    if "BI-004" in by_id:
        pc = by_id["BI-004"].get("pagination_collection", {})
        validations.append({"validation": "BI-004 전체 페이지", "passed": pc.get("is_complete", False)})
    if "MT-001" in by_id:
        validations.append({
            "validation": "MT-001 영상 공식 페이지 참고 링크",
            "passed": any(item.get("link_type") == "video" for item in by_id["MT-001"].get("supplementary_links", []))
            and all(not item.get("download", False) and not item.get("indexable", False) for item in by_id["MT-001"].get("videos", [])),
        })
    if "MT-009" in by_id:
        text = by_id["MT-009"].get("content_text", "")
        validations.append({"validation": "MT-009 구버전 금액 제외", "passed": "5천만원" not in text and "5천만 원" not in text})
        validations.append({
            "validation": "MT-009 웹툰 공식 페이지 참고 링크",
            "passed": any(item.get("link_type") == "webtoon" for item in by_id["MT-009"].get("supplementary_links", []))
            and not any("webtoon" in image.get("url", "").lower() for image in by_id["MT-009"].get("images", [])),
        })

    validation_df = pd.DataFrame(validations)
    validation_df.to_csv(paths.quality / "regression_tests.csv", index=False, encoding="utf-8-sig")

    service_action_count = sum(
        1 for document in page_documents for action in document.get("actions", [])
        if action.get("action_type") != "download"
    )
    download_action_count = sum(
        1 for document in page_documents for action in document.get("actions", [])
        if action.get("action_type") == "download"
    )
    pagination_detected_count = int(quality_df["pagination_detected"].fillna(False).sum()) if not quality_df.empty else 0
    pagination_complete_count = int(
        quality_df.loc[quality_df["pagination_detected"].fillna(False), "pagination_is_complete"].fillna(False).sum()
    ) if not quality_df.empty else 0

    summary = {
        "run_id": paths.root.name,
        "manifest_count": len(manifest_df),
        "target_count": len(target_df),
        "page_document_count": len(page_documents),
        "attachment_document_count": len(attachment_documents),
        "document_count": len(documents),
        "chunk_count": len(chunks),
        "status_counts": results_df["status"].value_counts().to_dict() if not results_df.empty else {},
        "quality_counts": quality_df["quality_status"].value_counts().to_dict() if not quality_df.empty else {},
        "service_action_count": service_action_count,
        "download_action_count": download_action_count,
        "attachment_count": len(attachment_manifest_df),
        "downloaded_attachment_count": int((attachment_manifest_df["download_status"] == "downloaded").sum()) if not attachment_manifest_df.empty else 0,
        "indexed_attachment_count": int(attachment_manifest_df["indexable"].fillna(False).sum()) if not attachment_manifest_df.empty else 0,
        "video_reference_count": sum(len(document.get("videos", [])) for document in page_documents),
        "supplementary_link_count": sum(len(document.get("supplementary_links", [])) for document in page_documents),
        "attachment_review_count": int(attachment_manifest_df["needs_review"].fillna(False).sum()) if not attachment_manifest_df.empty else 0,
        "pagination_detected_count": pagination_detected_count,
        "pagination_complete_count": pagination_complete_count,
        "non_paginated_count": len(page_documents) - pagination_detected_count,
        "regression_failed_count": int((~validation_df["passed"].fillna(False)).sum()) if not validation_df.empty else 0,
    }
    write_json(paths.quality / "quality_summary.json", summary)

    archive_path = Path(shutil.make_archive(
        base_name=str(paths.root), format="zip",
        root_dir=paths.root.parent, base_dir=paths.root.name,
    ))
    return {
        "paths": paths,
        "manifest_df": manifest_df,
        "target_df": target_df,
        "results_df": results_df,
        "quality_df": quality_df,
        "validation_df": validation_df,
        "attachment_manifest_df": attachment_manifest_df,
        "supplementary_df": supplementary_df,
        "documents": documents,
        "page_documents": page_documents,
        "attachment_documents": attachment_documents,
        "chunks": chunks,
        "summary": summary,
        "archive_path": archive_path,
    }


# 정책 불변식: 영상과 웹툰은 공식 페이지 참고 링크 전용입니다.
def validate_media_policy(config: PipelineConfig) -> None:
    if config.download_videos or config.download_webtoons:
        raise ValueError("영상·웹툰 다운로드는 이 파이프라인 정책에서 허용하지 않습니다.")


## 3. 42개 URL 최종 정책 CSV 생성

In [ ]:
import base64
import gzip
from pathlib import Path

RUNTIME_ROOT = Path('/content') if Path('/content').exists() else Path.cwd()
POLICY_CSV_PATH = RUNTIME_ROOT / 'KDIC_42_URL_RAG_Action_Media_Attachment_Final.csv'
POLICY_GZIP_B64 = """H4sIAFU0WGoC/+1cbW/bVpb+XqD/gfCHRQywlt+StAtsF2mybY1pJqmTLOabIEu0o4lMKSSVNIt+UBw5cGJn4szYYyWRvMrUTuxC3ZFl2VEwLgbozxFJzF/Y83IvRUq0Y8ty0s4UKBqZoi7vvec55zzn5fIff/u7XZtzCw3VrjScfDE6dkF1VmbtSj1qP87bm0tOCb76ftN9UvRfoHvt72dV9w8Fp1R3XuWiTrHsFpZVt5B3VitRZ67gvHrSrFWd8rLa3F1wVtejzk7R+cusvEb/RO3tXLP2ozp+7gv42IBxozC+/WjLmd9SvU9Ru7bkFPPqubiVTOtR+9Vd925FjiouXr2T0eQN8EPnZUm159ectSe+GdKc4KnO1qzq1Dbs3Zy7sOCU9qLO80V4On8f/AYnFvqLmYrzbNNZXXTKxeb2jrylcNe5dzfq1Ar2y4q89vyNO1dtu1bas6s4LfuvDZqZ/0t7u+68qDo/FtzZRadRjsLWO/NFuNTcqnd8yXthw3i1DdjTnHu/DMsvNGsV+Vfz9V5zq6o6u0V4SvTa+FcffjCkXrj80eDgEMx3rtmYg1XAsIABWAvIeJ+r/Ie7TH+YVsxKxqOZ2JSmJvV4KpvQopPZVCr4B0ACVqdeCvxWYTkrTjnv1Orq71TVXa7b828UZ+W+U1pQ7d05dznvPFuCr/S0rrX/b1qzYomYFYum9dSdKFxKaPG0AdO5pak41o4zX1F4zgqvXWG4Dii4quUXP722t/OIZZrUT69BYs1tuLdRddZyTgk+7VRwj8t3neKe/aKo0J7XB9ThweEzHw2e/WhoVHEAWuWiwohRvrx68SuFt1u9blkZ898jkdu3bw/cSCTjA2lj4IYRMTORRMa0jIyRtiKX4X9X7pgW/vvVtGVGTC2lxa0rcUMfSKQ//GCY5TN8JPmEXz2ioEIHUZrVoj27CDu1+sR+/BSV5WXlhOQnnoiqUCoozqs5lB18shdyoAdtAu29RNolMcKSGOlCU2DGgCzlU0Jdccd9tgA2A/+sFnFzutIgMWZgRCe/Bob4p9e4PUIml07hlL5bUNACPqsDhu0/FfvVRPq2nkrHEj4RtT5dUmNZKx2Np2KmmZy8E53KJmESKCM1kTRgT6LpyclkPBlLRbNGKpqcjMIKJlJaVEuZWtRMZ424xqs5rKj72BA0t/cUd3mj+TcQ98oTZzYHS1kuNhs5JWQRil3Luw/3UEv95lhhwy/01SnnwDGAum/vwGYrbNwVvtMp5RS7uuWsLOFogBAEDBhyoem+R7nLheZ2WXGW5+yZupiL/WqBPj1+qnjOCe6zN6uq0vIIK4vN2oL9cAl/i0AGQ1Jq4Gyc+ZKYhgJ2WLELi3iX+AJ+A2Ze8dyVvb6nsHMZ6Osd0BnffgN0JTttENZHGeujx8c6fHb/OHdMrIOZfiNsdHBY9IdPcwL3J+VGQJnclTX70RKJgWYG4wcnYs9vACshxNV+cGfeDJykmK4aU/BvIt6S12mW1+kjyasPMN2slpz8npuvgoQ+P/c1/B9pyFzBLr+BX7j3d9zlTWe+DIs6haAu1d2HDcXezdvrBZQy7Lrz5gdVrhC0A3EPesVP5Mf196mTsZuHkTbOgO08SHx+BwjhvjJFgc4VTtEv6JaWtgi15O3u7871tNa6uWQ/qMM60Ta1LbOORsJeB+aWI8kTV1bce8Xm6wbpfNi2452w8WAMjoiRyaRu+kESn45MTEi+8Hns5m+N2LR1LpO6Q4g4w4g4c3zegLj45QkQ4ov5ssI0E9jKTMV+kcOVDIj5Ae9T7Mez9ssfSWPvgvO8B6qwdQJEAmTTziXOsnTOHkc6vLZTsKvO00a/amiTmqHp4Hnj6elMzEiaaV3VviEpedLyblJFsMUxhPRgD9eUSyoPKGkzGN7mziLYQPR3TJYVb5TW73ptdM+gt5YEnvSKOeBdp/xEMD/2qnl7fk7oH7B2dO7oyO89UJxnVSdfduaKaJfBcNsz/+fsLqEb5yVJQrmySJ6cPPaAejGmJyc10/rptfTRHigh8gX3HeGI4C04SCQjE/+TmUjpnvmeiBuX8W/J8y/dMkyCwsfqZ2O+AAxlu1KAD4JtHO0y2G+21kgfYKawVAzO1wtHJv88CnAVjuVRXfzjodcNssx8Ebfr580yhwaJBPrZoo9EsrLzdbT2cJP9/YYIMAAnAMRqs1b9lUyGGbsLGXN6bEJ4oMtp07wQn9Y7YqhPGOzDxwU7oRNWMedUNxDzod/z1ypML66Z5hHj3v0e1gI8Xe5XYxlYsNqB7u4CXn4Es1n3ed6TrY+HKZzmkSI8KQnCprULb2iQpTdyPOlhqms195/dpSO8UcBAQ0AxU4Uvm40H4kLO+8vLGpyguKYm9Wg6o+nXjJQ0IyuLYCpkZABKOl9xH1fcfEOCCLSTTM3ukrNbPAnZ/dc3mZTeIbghFtxor3yM7xJteEguQk3c0WPTIN9UOn0jm/GECszEApMeEHI0k51IJeM+WTMhFHY2mOJ4UYVPsDdVNNGnPauGIqav+lXxxJZwBdVESlm+K0doJ53d8cx41rgMAAbTX7C3d5TLl65c5bRDYAE0WbK7JF7hAZImUbWUZmn/cdXIauSOltkXgGlfrZxYEPkZIOULWAAGkp/rpsUoGVav/ZaYCMeFgUhQAPigrwLXhMWirY4I8H+q8K/9Nx45HxB4MOk4sPu/zAIlgX/pOXZ1064tt2s+fEuTATPhozHCHnzLkPnW0FIxS0tETc24lQSSjLn5XSA6vbAWnOFCm9RpN1DkwbQVGQiBE0ARbKeIRIX5JxZQLgJRP4GAhSLJMf0mOgL6/EU20WlURhguwz2AS8C/gTu/v4DGHRM7AYl6FFcGQoowPCBr9+mS2F5hvLrNdh8wpsSUszpHtYHajxBkKIAw3639ajytW7G4Fcp5u2MFNDpV5sDFKZaWEiDAvzgCcv5cp+Q8xSZ492JREfNA3Lj3cvajKmY08lVfVvR4+YdsLBIDgEwhNi4ghtCaXNVSX2fNcMCMMmBG3gNguoJEcBJkY+gRhQ0b+T6Wiwpr7lKe/1LCrY9/ptIGBa1Ob8hIkCJWH4Nnzrt/fgCPZA89U3dWfpC+T9iQ9fBkee9gwYbEmjLQkoxZRuKidT0Rio3TjI3Rk/M9oNPO/UcUMBGfwAzCaoHvOrKxOGCsNlsB1sG7uY2g9ETuMhalh5JX8MWPFGcKCfp5E8IYFabUABcKZI5+J6kJJeCWHzqlhRNwLV8aGe3LpInepQMDZ9SLVzkTUt0EzXLuv0BpArNynxakoI/2FSwFgnSRyQII+O7w3yByXRCVqLeSCS19aBz0yRIl2Z+2arbCHQCUGAia4L7DZstaOT2SvZcPaWU9DgJHFmhM1NBuZjXTMqO0sv03SQkuheeORL/ZqKphy8R1jZ/7gvIi7ZBrLdyX9wBTBBszU/FBDMzOZnW/PAWIq0HpOjZWxIp7EvLemDZM+h9apA4UnmUUDr8PFHaVrwiMcdqe38AOhmC2ooOwhATDYBBy9sP1XqYwfCUTYYaFu4G5YA3pZQXraF15nbdINTR98TFLduQ9SNZHDY5OTD26gQNU9iMa3n2KJBvy9mLeeeOLdOIxlIHafr9CytcTwto+slBwVmT0PXJmq+tgUnxspFm9C//1gJS8BR5IR5CKdEDkE4bIaJcQCS2o7ocK8RusrLWXaQgBokJzUFFNaC5Kzhfnri5SJgzigg0pdVjnZDR+XYvfUDHwhfD34RpabnupamPWYX7Nfpk/4ZLc1UuXhwbJJKw9sbd37MUCltQhwsbAZbsMgPd6fDAtThYLnkcfAQqixkoxb8uFkLuhX4kSvBidfMVWtfuCztuqrRfNhPHFrYkbXsF1eJCxc7qX2IE9U4YGD1t9pR1W/knggsPy9MH1EyS41gfP59X5m4W66bl4m4SvpjNDgyzaIRbtmR54joBfCBoHnOvrDbSbn1KbSalAbOmHI3sMbxz/KD5PASKq1fEuCJm+W1BgPhAhSn/hlJ/AdaonZ/VEy1/wXb32Ff5Rqebmm4OfLQhmzZUR+4+Exq7oYLvUk0YklkgYphY5R/+csyz9omUZHX2Yw4yBsyeAAS97wT7dS1sECq5eLiMMMWrMsmLx69OabnUBE/9zwCCUsNaOm9uodgazv4Qy7yhFMDTTjsIuVWMhUi6v2dXn6D3OJdITmjKuxRKa4Q+j2X/8y9Z6w/REpnXQ97G2eIXesKTO8AirzMc/K5XpUlmCQ+zTFfELbIkYHjyKsgR7IUXRS/TE/6oq+6hKxjhYTUZZTT55h2oSDF+PkHaITiZTlmZoCS88OSWKZMzM+9U+PvWjBDIB1BLqHeYhnanlQVEUPhEjTmNgqw1+BUJClZL39/Uqf3FgOu0omTQzC78i5TwwocYbwCsUmbTg5jCtIZS3pdBaO8RbgwQYWwmWN+zFsgyLCPM0fGeu7dL5cXTlYlROM9ZPONV2oCJYhonx0pWMYYVlZ4ZPkxZAsPMOnUU9Z5c34IpdfkM1k+brIp5HS2lTIF9DEwY1CVD45lCZOBoJIk0aRpgyeq5o2Qm0qV861X4/GM4KaJB8vNeq6SGabaF7bxFlhv1BPJQ4rNZV+WDnB9QbmgjXAjgCpQoBgGWmogRm11E6ijB0xCnAXhWR2pEzrqW+it1Oj09lUx3A4bLB0NA7BA42/T/ddPJl915JTcX0RFKfihrpLFjGcJzIrhdhNKWnJDygulFJCYZUeEzfNy2rSSd5vFBddjv89JpDqH61FbR/y1E8R3CqrAlSYlh0KPUmz/cKbNksVps3yWEDEskivXrVytcwSBTMBz6rYmajBRTUvpeVHgPlipW1zl/vtC6c1R8afqc+thXKdlmA9gfD7DJA7M/qWF+ZLzar2Nj+3R6mUP2HvUIgEspEO/DiUdNDIeZ9HBKjZhklDZuVjN/AvozbYJvTtwdkkmLgOmAe84kCdsD6Wt00vxLU/ZTmv01rPJ4JJ6dcNBkaOeHU1/GTXlwUod4drDQiGUUJFMt0eAwkYa9jZXIO9/TAshi2TGDrRLBXA+jxFDACs0edvqIwQueMfUdbxYkHPuAwoIhTLIIdy7vpYMPx7eY0hSYiQtk37cUVkaHRdyH/kJoZKl2j6h1rYPk6KxVqy3u6jNF8Uge3G4vjzkZ9ietUUr/B5qjNCUdjZBwO8MV9fiuKHgvwyHmxZ5uiahB0zS1dl72eGML8VbxFoK/NLvtNb/BZZIw8YCHDc+4/kt3sJdmI2KcyvLGzBE8HrT7BeR2zkRX0ASlqq1UVttr+3wfiVFSw4CINVKMKAVdI0NCrmksLpIxKrLV8nZqMjyV06864meK+1REuvAydfm91XQJlwGAdXNCT2k8Bq/cmAthOd7kYtF37WqoQN9/p0U+O+smTWYxMmExtU56TKXnHIUUhDgRfLIfV7qjDqEcHrsIqvNiP2HGIf0i9cI6bjLbyeKoMX+6w7DW8v/XaagWtsGfCfN+FvGThLSjwd9dzqNU5XlvM2IGDe3ft+XoLAfG0bmZTlhT//rW744mftnCEarngf2VLj2xcB2OEtLQG8+KXtWCfqj1focfzCUJfuCjOUxO3SmQzwOxilhadgmgqQ0cOd3Ng7nuGE2MikppIWbHfT1uRr+DDOfiAKQnvc9iLH4Z5wcO9xAzF2rIr7x979xUWpbBJQK2va7pFe9FqR9/XpR0UV4qudx5eYSfGkQrV28h2dQSacv6+n0qIiVZ5gTRJh6RHorftKB7w+rttlceBxFilOmU6fNvmrcZ33jyse/EdYGZMn0yP6TcvmHFTR5vTgZ0RoSxdYeeoJPg34/92PmBEBGekCJCKe83qIkbr3qnen6NhGSLDQmctMZZa9N74wlZDwYszBTIOMxusWkBCFimByXew2aGOoh2givnjdQ8lkhEDgtakpUXwGO9vDD0Ooo8BLNqFPcoLGO2NoYCtf7YJGkhelK6qscTvY3GqW6XBUN5RDe1WUrt9AM1oG0BhX01d4RgGk/4g5Qh0Cz5tIL0WOiN8fK0AxkT7Bqi2DhTaC+05OJLGQP6yUW3WcwJ7PYiU5CKwk+35IrUxi7nB9uXp/La7MkfkXL6WLMJ3iJkIisoc5FgU9ZCm4byhJcbjt4w7+EcHTE4zTE73BCaYP5ipglLj+8KOhA/uAOffg6umEVrvdxLpJpTeyhydNAkwUikBgRxOilxSWzUVLCclsoYmG517Cgp6Z5qCaev1smhhB+HimzfezMoKCq1O3sMuTHQUApcHZ7fohTQ1sCS8ei5AYcIIVnziKLls6p91RNwjZxgdZ3qCDiHjZwsQx3WFDvply2zQo7y3akiMtHuTnz04vMQKZVQw6wdMw11asFcLzdcNQUrfhfzHpzrkf5blf/bw8udoFUTUmOOkOuZSgDdv5WFZWH19vhgJHFThQo+gmhnm3PiyxOMQTbta9E4C42u7UH5eS5GvY7mda7ZNFOPg9pkqgXrvt0nTzGrtxFP0pbWI53vIT3tvCig6u0Xsp8Md8WWE2kmqL5cSONvvy6X8M+Wsexfqm6A8WNuKoBZ9ricuaBPWhZQ+hQw8LIs98jHr1MeH1qnwTm5fDCIOjmFn7sEkrOM3h27Q9QVh3pkxecas1X95wu9RavXy05NbkaMMzwpr2Phk/2mLiDXhp9XHLVJB4kVL1NLrKxTzGfeT6tD2hWcMgk/UL8WrTks5e/4BPByID2oeCDv82n6GFRa6Neus5hEz/CJHsln8FXcVYC7YPyYPybQeu90JWUcucnRO0jO4jyv48lJ8D4b4ovMYPFz1ihui3CzQZWiZtGG1ihwnFuB501QwIwmruDbm6/rhpkvv4CLtGJ1lDRxb7Tgs35sTIu1G5bwen74QTxlhBmV0kLE0fGgshRuUDoAc/0Vr7U3+MKiU7/tr5vehE9/kWM2hNVxdBzSKtx0jUSRn4T/LQcKVLqrH3f0o3ctGxkIJs0iHWKQj+4iU6vCUkwXKMgeqhVoEAanY4ePais4x22yFaDRokXhJ1trcj6kBV8MUaiI9HUvqrRe9ceJVBshY5vI/Uz4rLOushOwINTzQGb4+InEdIx0GdO+1m8CKGVOaFZ3ImkkdGx9B973zBm8Ttz9zDZpS2HOXN+WL58DwYvDwK3M7jJEdu2VOxlLXdCM1lppK7Wtuh1k3Rw9tbkM0jx2fW8i3HJ9yyn8KXpgTWecFiyde52F6neKyotMKkIR6/U4NjMQ/ls5WPNMXCPik10pYo5PunGO/6nllOSrnx7n8rQTjoOB6jpeGDzgA0dwVskqFO8Akhl9WEMZA/coP3IeYZZHlQDkZ2Q3Z/l4SCnrYk/cIb7fS8Qi+fQCgFDkfMzTNGNemkqYlOhdT8PHDD/4faxTPQfZgAAA="""
POLICY_CSV_PATH.write_bytes(gzip.decompress(base64.b64decode(POLICY_GZIP_B64)))
print('정책 CSV:', POLICY_CSV_PATH)

## 4. 실행 설정

기본 설정은 42개 전체 수집과 공개 첨부파일 실제 다운로드입니다.

처음 다운로드 기능만 시험하려면 `run_only_url_ids`에 다음처럼 넣습니다.

```python
run_only_url_ids=['MT-001', 'MT-009', 'BI-001', 'DP-003']
```

In [ ]:
from kdic_final_pipeline import PipelineConfig, run_pipeline

CONFIG = PipelineConfig(
    select_business_functions=[],
    run_only_url_ids=[],
    max_urls=None,

    request_delay_seconds=1.2,
    request_timeout_seconds=30,
    max_retries=3,
    respect_robots_txt=True,

    enable_generic_pagination=True,
    pagination_max_pages=100,
    pagination_page_size=10,

    # 공개 첨부파일 원본 수집
    download_direct_attachments=True,
    download_js_attachments_with_playwright=True,
    extract_attachment_text=True,
    create_attachment_documents=True,

    # 영상·웹툰은 정책상 다운로드 금지
    collect_supplementary_media_links=True,
    download_videos=False,
    download_webtoons=False,

    # 일반 이미지는 기본적으로 다운로드하지 않음
    download_images=False,

    create_chunks=True,
    chunk_max_chars=1600,
    min_chunk_chars=80,
)

print(CONFIG)

## 5. 실행 전 정책·파서 자체 테스트

In [ ]:
import pandas as pd

from kdic_final_pipeline import (
    PipelineConfig,
    normalize_review_manifest,
    parse_html_document,
    structure_aware_chunks,
)

policy_df = pd.read_csv(POLICY_CSV_PATH, encoding='utf-8-sig', dtype=str).fillna('')
manifest_df = normalize_review_manifest(policy_df)
assert len(manifest_df) == 42
assert manifest_df['url_id'].nunique() == 42

# 영상은 공식 게시 페이지 참고 링크만 생성
video_html = '''
<html><body><div class="contents">
  <video src="/media/guide.mp4" poster="/media/poster.png" controls></video>
  <p>제도 본문 설명입니다.</p>
</div></body></html>
'''.encode('utf-8')
video_row = manifest_df[manifest_df['url_id'] == 'MT-001'].iloc[0].to_dict()
video_doc = parse_html_document(video_html, video_row['source_url'], video_row, 'utf-8')
assert len(video_doc['videos']) == 1
assert video_doc['videos'][0]['download'] is False
assert video_doc['videos'][0]['indexable'] is False
assert any(item['link_type'] == 'video' for item in video_doc['supplementary_links'])

# 웹툰 이미지는 이미지 목록·RAG에서 제외하고 공식 페이지 링크만 생성
webtoon_html = '''
<html><body><div class="contents">
  <p>현재 절차 설명입니다.</p>
  <img src="/assets/webtoon/webtoon01.jpg" alt="안내 웹툰">
</div></body></html>
'''.encode('utf-8')
webtoon_row = manifest_df[manifest_df['url_id'] == 'MT-009'].iloc[0].to_dict()
webtoon_doc = parse_html_document(webtoon_html, webtoon_row['source_url'], webtoon_row, 'utf-8')
assert any(item['link_type'] == 'webtoon' for item in webtoon_doc['supplementary_links'])
assert not any('webtoon' in item['url'].lower() for item in webtoon_doc['images'])

# 짧은 FAQ도 삭제하지 않고 청크 생성
faq_html = '''
<html><body><div class="contents">
  <div class="accoWrap"><div class="accodian"><dl>
    <dt>질문 신청 기한은? 열기</dt>
    <dd><p>답변 1년 이내입니다.</p></dd>
  </dl></div></div>
</div></body></html>
'''.encode('utf-8')
faq_row = manifest_df[manifest_df['url_id'] == 'DP-005'].iloc[0].to_dict()
faq_doc = parse_html_document(faq_html, faq_row['source_url'], faq_row, 'utf-8')
faq_chunks = structure_aware_chunks(faq_doc, PipelineConfig())
assert sum(item['chunk_type'] == 'faq' for item in faq_chunks) == 1

print('Manifest 42개 검사: 통과')
print('영상 참고 링크 정책: 통과')
print('웹툰 참고 링크 정책: 통과')
print('짧은 FAQ 무손실 청킹: 통과')

display(manifest_df[[
    'url_id', 'page_title', 'attachment_download_policy',
    'attachment_rag_policy', 'video_policy', 'webtoon_policy'
]])

## 6. 42개 전체 파이프라인 실행

In [ ]:
RESULT = run_pipeline(
    review_csv_path=POLICY_CSV_PATH,
    runtime_root=RUNTIME_ROOT,
    config=CONFIG,
)

print(RESULT['summary'])
display(RESULT['results_df'])
display(RESULT['quality_df'])
display(RESULT['validation_df'])

## 7. 영상·웹툰 참고 링크 확인

In [ ]:
supplementary_df = RESULT['supplementary_df']
if supplementary_df.empty:
    print('생성된 참고 링크가 없습니다.')
else:
    display(supplementary_df[[
        'parent_doc_id', 'link_type', 'label', 'url',
        'display_condition', 'content_status'
    ]])

## 8. 첨부파일 다운로드·인덱싱 정책 확인

In [ ]:
attachment_df = RESULT['attachment_manifest_df']
if attachment_df.empty:
    print('발견된 첨부파일이 없습니다.')
else:
    display(attachment_df[[
        'parent_doc_id', 'display_name', 'download_method',
        'download_status', 'processing_policy', 'indexable',
        'official_download_url', 'user_action_url', 'sha256',
        'needs_review'
    ]])

    print('첨부파일 처리 요약')
    display(
        attachment_df.groupby(
            ['download_status', 'processing_policy'],
            dropna=False,
        ).size().rename('count').reset_index()
    )

## 9. 주요 결과 파일

```text
processed/documents.jsonl
processed/chunks.jsonl
processed/attachment_manifest.csv
processed/attachment_manifest.jsonl
processed/supplementary_links.csv
processed/supplementary_links.jsonl
quality/attachment_review.csv
quality/quality_report.csv
quality/regression_tests.csv
```

- `supplementary_links`: 영상·웹툰 공식 게시 페이지 참고 링크
- `attachment_manifest`: 내부 원본 다운로드·해시·텍스트 추출·인덱싱 정책
- `actions`: 사용자에게 제공할 공식 신청·조회·다운로드 링크

## 10. 결과 ZIP 다운로드

In [ ]:
archive_path = RESULT['archive_path']
print('결과 ZIP:', archive_path)
print('크기:', f'{archive_path.stat().st_size / 1024 / 1024:.2f} MB')

try:
    from google.colab import files
    files.download(str(archive_path))
except Exception as error:
    print('자동 다운로드가 시작되지 않으면 왼쪽 파일 패널에서 직접 다운로드하세요.')
    print(error)